# I

## Intro to Data Cleaning with Apache Spark


- Course focus: **data cleaning** with Apache Spark and Python
- Covers: what data cleaning is, why it matters, how to do it at scale with Spark

### What is data cleaning?

**Data cleaning** = preparing raw data for use in **processing pipelines**. (Pipelines will be covered later — for now, think "the series of steps that take raw data to finished analytics".)

Why it matters:

- If data isn't clean, it isn't **trustworthy**
- Dirty data breaks downstream pipelines in subtle, expensive-to-debug ways
- A necessary part of any **production** data system — not an optional polish step

**Typical data-cleaning tasks:**

- **Reformatting or replacing text** (standardising dates, fixing casing, stripping whitespace)
- **Performing calculations** on the data (deriving columns, unit conversions)
- **Removing garbage or incomplete data** (bad rows, nulls, out-of-range values)

### Why perform data cleaning with Spark?

Two recurring problems in any data-cleaning system:

1. **Performance** — handling large volumes
2. **Flow organisation** — co-ordinating multi-step pipelines

**On performance:**

Traditional languages (Perl, C++, standard SQL on a single box) are fine for small datasets. They fall over once the data stretches into **millions or billions of rows** — single-machine throughput just isn't enough.

Spark's edge: processing **scales out** across a cluster as the data grows. Capacity expands with the workload rather than being capped by one machine.

**On flow:**

Large-scale cleaning naturally splits into a **pipeline** of stages — read, validate, transform, deduplicate, enrich, write out. Spark handles all of that within a **single framework** — no need to glue together half a dozen separate tools.

### Data cleaning — example

A small motivating example. Given raw data:

| Name | Age (years) | City |
|---|---|---|
| … | … | … |

The requirement for the cleaned output:

- **First name and last name in separate columns** (split a single "Name" field)
- **Age in months** instead of years (unit conversion)
- A new **state** column derived from the city
- **Out-of-the-ordinary rows removed** (bad or garbage entries dropped)

All of that is achievable with Spark transformations in a single pipeline — the cleaned DataFrame then flows into whatever processing comes next.

### Spark schemas

A primary function of data cleaning is **verifying data is in the expected format**. Spark provides built-in support for that via **schemas**.

Important to lock in: a **schema** defines and validates the **number and types of columns** in a DataFrame — analogous to a schema in a database or XML.

**Supported field types include:**

- Integers, floats
- Strings
- Dates
- Arrays
- Map structures

**Two big benefits of defining a schema:**

1. **Correctness** — Spark can filter out non-conforming data **during read**, catching problems as early as possible
2. **Performance** — without a schema, Spark **infers** one by reading the data **twice** (once to figure out types, once to actually load). An explicit schema skips the inference pass → **one read** instead of two

### Example Spark schema

Defining a schema uses the `pyspark.sql.types` module.

```python
import pyspark.sql.types

# Build the schema as a StructType of StructFields
peopleSchema = pyspark.sql.types.StructType([
    # name, dataType, nullable
    pyspark.sql.types.StructField('name',    pyspark.sql.types.StringType(),  True),
    pyspark.sql.types.StructField('age',     pyspark.sql.types.IntegerType(), True),
    pyspark.sql.types.StructField('city',    pyspark.sql.types.StringType(),  True),
])
```

With shorter imports — the common idiom:

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

peopleSchema = StructType([
    StructField('name', StringType(), True),
    StructField('age',  IntegerType(), True),
    StructField('city', StringType(), True),
])
```

Apply it to a read:

```python
people_df = spark.read.format('csv') \
    .load('people.csv', schema=peopleSchema)
```

Key points on `StructField`:

- **Field name** — the column name
- **Data type** — `StringType()`, `IntegerType()`, `DateType()`, `ArrayType(...)`, etc.
- **Nullable flag** — `True` if the column can contain nulls

Important to remember: `.load()` takes both the **filename** and the **schema** — that's where the schema actually gets applied against the data.

### Key takeaways — cheat sheet

**The data-cleaning pitch for Spark in one table:**

| Problem | Why Spark helps |
|---|---|
| Huge datasets exceed single-machine limits | Scales out across a cluster |
| Pipelines sprawl across many tools | Single framework for read, transform, write |
| Dirty data slips through on read | Explicit schema filters bad rows at load time |
| Schema inference is slow | Explicit schema → **one** read instead of two |

**Defining a schema — minimal template:**

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField('col1', StringType(),  True),
    StructField('col2', IntegerType(), True),
    # ...
])

df = spark.read.format('csv').load('file.csv', schema=schema)
```

**Things to lock in:**

- Data cleaning = **preparing raw data for pipelines** — not optional in production
- Three broad task types: **reformatting, calculating, removing garbage**
- Traditional single-machine tools break down at scale; **Spark scales out**
- A Spark **schema** = `StructType` of `StructField(name, type, nullable)`
- Schemas give both **correctness** (filter non-conforming rows) and **performance** (skip inference → single-read)
- Types live in **`pyspark.sql.types`** — import `StructType`, `StructField`, and whichever leaf types the data needs

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("pyspark-cleaning").getOrCreate()

26/11/20 18:46:41 WARN Utils: Your hostname, DDG-1000 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/20 18:46:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 18:46:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Import `*` from the `pyspark.sql.types` library.
from pyspark.sql.types import *

# Define a new schema using the `StructType` method.
people_schema = StructType([
  # Define a `StructField` for `name`, `age`, and `city`. Each field should correspond to the correct datatype and not be `nullable`.
  StructField('name', StringType(), False),
  StructField('age', IntegerType(), False),
  StructField('city', StringType(), False)
])

## Immutability and Lazy Processing


- Two deeper Spark concepts that explain **how** DataFrame operations actually work under the hood
- Covers: variable mutability in general, **immutability** of Spark DataFrames, and **lazy processing**

### Variable review — mutability in most languages

In Python (and most other languages), variables are **fully mutable**:

- Their values can be changed at any time, as long as the variable is in scope
- Very flexible — but creates problems when **multiple concurrent components** try to modify the same data

Workarounds in traditional languages:

- **Mutexes**, **semaphores**, locks, and similar synchronisation primitives
- These add significant complexity, especially for non-trivial programs
- Tricky to get right; a common source of deadlocks and race conditions

### Immutability in Spark

Unlike typical Python variables, **Spark DataFrames are immutable** — a design choice drawn from **functional programming**.

What this means in practice:

- A DataFrame is **defined once** and **cannot be modified** after initialisation
- If a variable name is reused, the original data is **discarded** (unless still referenced elsewhere) and the name is reassigned to a new DataFrame
- **No in-place modification** — every "change" produces a new DataFrame

Why on earth is this efficient? Because immutability lets Spark **share data safely** across every component of the cluster — no need to worry about concurrent writes clobbering each other. The complexity of mutexes and locks simply goes away.

### Example — immutability in action

Load a CSV, add a column, drop another. The key thing to notice: the DataFrame isn't being *mutated* — each step produces a **new** DataFrame object that gets bound to the same variable name.

```python
# Step 1 — create a DataFrame from a CSV
voter_df = spark.read.csv('voterdata.csv')

# Step 2 — add a 'fullyear' column (2-digit year + 2000)
# This doesn't mutate voter_df. It creates a NEW DataFrame
# definition carrying the transformation and rebinds the name.
voter_df = voter_df.withColumn(
    'fullyear',
    voter_df['year'] + 2000
)

# Step 3 — drop the original 'year' column
# Again: new DataFrame, name rebound. Old one is destroyed.
voter_df = voter_df.drop('year')
```

Important to lock in:

- The **original** `year` column is **permanently gone from this DataFrame instance**
- The **underlying CSV file is untouched** — can always reload into a fresh DataFrame if the column is needed again
- Every "modification" is really a **new definition** being swapped into the variable name

### Lazy processing

So how does Spark do all this quickly, even on huge datasets? **Lazy processing**.

**Lazy processing** = very little actually happens until an **action** is performed.

In the previous example — reading a CSV, adding a column, dropping another:

- **No data is actually read, added, or modified at those points**
- Spark only **updates the instruction graph** (the chain of transformations) for what to do *when asked*

This gives Spark room to **optimise the plan** before executing anything — reordering, combining and pruning steps to produce the most efficient execution path.

Nothing runs until an **action** is called. Adding `count()` forces execution of every queued transformation:

```python
voter_df = spark.read.csv('voterdata.csv')

voter_df = voter_df.withColumn(
    'fullyear',
    voter_df['year'] + 2000
)

voter_df = voter_df.drop('year')

# This is an action — NOW Spark actually reads the file,
# applies both transformations, and returns a result.
voter_df.count()
```

Key point: transformations are **lazy**; actions are **eager**. The transformation chain is just a recipe — it doesn't cook anything until an action asks for the output.

**Common actions that force execution:**

- `count()`
- `show()`
- `collect()`
- `take(n)`
- `write(...)`

### Key takeaways — cheat sheet

| Concept | What it means | Why it matters |
|---|---|---|
| **Immutability** | DataFrames can't be modified in place; "changes" produce new DataFrames | Safe sharing across the cluster; no locking needed |
| **Lazy processing** | Transformations build a plan; nothing runs until an action | Lets Spark optimise the whole chain before executing |

**Things to lock in:**

- Spark DataFrames are **immutable** — every transformation returns a **new** DataFrame
- Reassigning to the same variable name **replaces** the old definition (original is garbage-collected if nothing else holds it)
- The **underlying data file is never touched** — only in-memory DataFrame definitions change
- **Transformations** (e.g. `withColumn`, `drop`, `filter`, `select`) are **lazy** — they just extend the plan
- **Actions** (e.g. `count`, `show`, `collect`, `write`) **trigger execution** of the plan
- Immutability + laziness together → safe concurrency **and** aggressive query optimisation; the two pillars of Spark's model

In [3]:
import pyspark.sql.functions as F

In [4]:
# Load the CSV file
aa_dfw_df = spark.read.format('csv') \
    .options(header=True) \
    .load('AA_DFW_2018.csv.gz')

# Add the airport column using the F.lower() method
aa_dfw_df = aa_dfw_df.withColumn(
    'airport',
    F.lower(aa_dfw_df['Destination Airport'])
)

# Show the DataFrame
aa_dfw_df.show()

+-----------------+-------------+-------------------+-----------------------------+-------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|airport|
+-----------------+-------------+-------------------+-----------------------------+-------+
|       01/01/2017|         0005|                HNL|                          537|    hnl|
|       01/01/2017|         0007|                OGG|                          498|    ogg|
|       01/01/2017|         0037|                SFO|                          241|    sfo|
|       01/01/2017|         0043|                DTW|                          134|    dtw|
|       01/01/2017|         0051|                STL|                           88|    stl|
|       01/01/2017|         0060|                MIA|                          149|    mia|
|       01/01/2017|         0071|                LAX|                          203|    lax|
|       01/01/2017|         0074|                MEM|                           

## Understanding Parquet


- Spark can read text and CSV files, but **CSV isn't always the best working format**
- Covers: CSV's limitations (general and Spark-specific), what **Parquet** is, how to read/write it, and using Parquet files with SQL

### Difficulties with CSV files

A few baked-in problems with the format itself:

- **No schema** — no data types, no column names beyond an optional header row
- **Delimiter collisions** — content containing a comma (or whatever the delimiter is) requires **escaping**
- **Escape collisions** — using the escape character inside content means yet **further escaping** (the escape-the-escape problem)
- **Limited encoding support** — available encodings depend on the language reading the file

### Spark-specific issues with CSV

On top of the general issues, CSV plays badly with Spark in particular:

- **Slow to import and parse** — text parsing is CPU-heavy
- **Cannot be shared between workers during import** — a single file can't be chunked across the cluster the way binary formats can
- **Schema must be inferred by reading all the data first** (if none is defined) — doubles the I/O cost
- **No predicate pushdown** — more on this in a moment

**Predicate pushdown**

A key Spark optimisation: **order tasks to do the least amount of work**.

- **Filter data *before* processing** → the main trick predicate pushdown applies
- Drastically reduces the amount of data that has to flow through the pipeline on large datasets
- **CSV cannot use predicate pushdown** — the whole file has to be read before anything is filtered

**Intermediate file representations**

Spark pipelines are often multi-step and benefit from caching intermediate results to disk. Using CSV for that means re-doing schema definition, encoding handling and parsing every single stage — a lot of wasted effort.

### The Parquet format

**Parquet** = a **compressed columnar** data format built for Hadoop-based systems (Spark, Hadoop, Apache Impala, …).

Key properties:

- **Columnar layout** → read only the columns actually needed
- **Compressed** → smaller files, faster I/O
- **Chunked structure** → efficient reads/writes **without processing the entire file**
- **Schema information embedded** in the file itself — no need to define it on every read
- **Encoding handled automatically** — no CSV-style escape headaches
- **Supports predicate pushdown** → significant performance improvements over CSV
- **Perfect as an intermediate / on-disk representation** for processed data

Important to lock in: Parquet is **binary**, so it can only be opened with proper tools — a text editor won't do. That's the one real cost compared to CSV, which anyone can crack open in Notepad.

### Working with Parquet

Reading is dead simple. Two equivalent styles:

```python
# Long form — using read.format
# .format() specifies the format; filename extension is ignored
df = spark.read.format('parquet').load('filename.parquet')

# Shortcut — generally preferred
df = spark.read.parquet('filename.parquet')
```

Writing mirrors reading:

```python
# Long form
df.write.format('parquet').save('filename.parquet')

# Shortcut
df.write.parquet('filename.parquet')
```

The **long form** is useful when extra option flags are needed — e.g. overwriting an existing file:

```python
df.write.format('parquet') \
    .mode('overwrite') \
    .save('filename.parquet')
```

Key point: the two forms are **interchangeable** — the shortcut is just syntactic sugar over `read.format('parquet')`.

### Parquet and SQL

Parquet pairs beautifully with Spark SQL — both because the schema is already defined and because predicate pushdown kicks in.

Workflow:

1. Read the Parquet file into a DataFrame
2. Register it as a temporary view with `createOrReplaceTempView()`
3. Query it with `spark.sql(...)`

```python
# 1. Read the Parquet file
flights_df = spark.read.parquet('flights.parquet')

# 2. Register as a temp view
flights_df.createOrReplaceTempView('flights')

# 3. Run a SQL query — e.g. flights under 100 minutes
short_flights_df = spark.sql(
    """
    SELECT *
    FROM flights
    WHERE flight_duration < 100
    """
)

short_flights_df.show()
```

Because the backing store is Parquet:

- **Schema is already defined** → no inference step
- **Predicate pushdown** → the `WHERE flight_duration < 100` filter is pushed down into the file read, so irrelevant chunks never leave disk

### Key takeaways — cheat sheet

**CSV vs Parquet at a glance:**

| | **CSV** | **Parquet** |
|---|---|---|
| Format | Text | Binary, columnar |
| Schema | None (or just a header) | Embedded |
| Compression | None by default | Built-in |
| Shareable between workers on import | No | Yes (chunked) |
| Predicate pushdown | No | **Yes** |
| Edit with a text editor | Yes | **No** |
| Good for intermediate storage | No | **Yes** |

**Parquet cheat sheet:**

```python
# Read
df = spark.read.parquet('file.parquet')

# Write
df.write.parquet('file.parquet')

# Overwrite an existing file
df.write.format('parquet').mode('overwrite').save('file.parquet')

# Query with SQL
df.createOrReplaceTempView('my_table')
spark.sql("SELECT * FROM my_table WHERE col > 100").show()
```

**Things to lock in:**

- CSV = fine for input/output, **poor for intermediate storage**
- Parquet = **columnar, compressed, schema-aware, predicate-pushdown friendly** → ideal for Spark workflows
- **Predicate pushdown** = filter early, in the file-read stage → huge win on big datasets
- Shortcut vs long-form read/write are **interchangeable**; long-form needed only when extra options (like `mode('overwrite')`) are required
- Parquet + `createOrReplaceTempView` + `spark.sql` = fast SQL analytics with minimal setup

In [5]:
df1 = spark.read.format("csv").load("AA_DFW_2017_Departures_Short.csv.gz")
df2 = spark.read.format("csv").load("AA_DFW_2016_Departures_Short.csv.gz")

In [6]:
# View the row count of df1 and df2
print("df1 Count: %d" % df1.count())
print("df2 Count: %d" % df2.count())

# Combine the DataFrames into one
df3 = df1.union(df2)

# Save the df3 DataFrame in Parquet format
df3.write.parquet('AA_DFW_ALL.parquet', mode='overwrite')

# Read the Parquet file into a new DataFrame and run a count
print(spark.read.parquet('AA_DFW_ALL.parquet').count())

df1 Count: 139359
df2 Count: 140605


279964


In [7]:
df3_renamed = df3.withColumnsRenamed(
    {
        "_c3": "flight_duration"
    }
)

df3_renamed.write.parquet('AA_DFW_ALL.parquet', mode='overwrite')

In [8]:
# Read the Parquet file into flights_df
flights_df = spark.read.parquet("AA_DFW_ALL.parquet")

# Register the temp table
flights_df.createOrReplaceTempView('flights')

# Run a SQL query of the average flight duration
avg_duration = spark.sql('SELECT avg(flight_duration) from flights').collect()[0]
print('The average flight time is: %d' % avg_duration)

The average flight time is: 151


In [9]:
flights_df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- flight_duration: string (nullable = true)



# II

## DataFrame Column Operations

- Focus: using **Spark column operations** to clean data
- Covers: DataFrame refresher, where operations live (`F` vs column methods vs DataFrame methods), common transformations, filtering, string manipulation, and `ArrayType()` columns

### DataFrame refresher

Quick recap on what a Spark DataFrame actually is:

- Made up of **rows and columns** — generally analogous to a database table
- **Immutable** — any change to the structure or content creates a **new DataFrame**
- Modified only through **transformations** (never in place)

Two illustrative transformations:

```python
# .filter() — keep only rows matching a condition
df.filter(df.name.startswith('M'))

# .select() — keep only specified columns
df.select('name', 'position')
```

### Where operations live — three homes

Spark operations split into **three families**, each with a different home. Knowing which lives where kills most of the guessing about syntax.

| Home | What lives there | Mental test |
|---|---|---|
| **`F.something(col)`** | Transformations that produce a **value** from a column | "Would this be a SQL function? `UPPER(name)`, `YEAR(date)`?" |
| **`col.something()`** | Questions/actions about a **specific column** | "Is this about *this column's* type, nullability, naming, or a per-value test?" |
| **`df.something()`** | Operations on the whole **DataFrame** | "Is this about rows, joins, grouping, or sorting?" |

**The `F` camp — SQL built-in functions**

Conventionally imported as `F`:

```python
import pyspark.sql.functions as F
```

Anything that in SQL would look like `FUNCTION(col)` lives here:

```python
F.upper(col)        F.lower(col)        F.length(col)
F.year(col)         F.month(col)        F.to_date(col)
F.split(col, ' ')   F.size(arr_col)
F.round(col, 2)     F.abs(col)
F.concat(c1, c2)    F.regexp_replace(col, r'\d+', '')
F.when(cond, a).otherwise(b)
F.coalesce(c1, c2)
F.lit(42)           # wrap a literal as a Column
F.col('name')       # grab a column by name

# Aggregates live here too
F.sum(col)   F.avg(col)   F.count(col)   F.max(col)   F.min(col)
```

**The column-method camp — metadata, type, per-value tests**

Called directly on the column object (`df.col_name.something()`):

```python
# Type conversion
df.age.cast(IntegerType())

# Renaming
df.name.alias('full_name')

# Null tests
df.name.isNotNull()
df.name.isNull()

# Per-value string tests
df.name.contains('Smith')
df.name.startswith('M')
df.name.endswith('son')
df.name.like('%Smith%')         # SQL LIKE
df.name.rlike(r'^M.*')           # regex LIKE

# Membership / ranges
df.age.isin([18, 21, 65])
df.age.between(18, 65)

# Array / map access
df.splits.getItem(0)
```

Plus **operators** — always inline, never via `F`:

```python
df.age + 10            df.price * 1.2
df.age > 18            df.name == 'Alice'
(df.age > 18) & (df.country == 'UK')
~df.name.contains('Smith')
```

**The DataFrame camp**

```python
df.select(...)    df.filter(...)    df.where(...)
df.withColumn(name, expr)           df.drop(...)
df.groupBy(...).agg(...)            df.orderBy(...)
df.join(other, ...)
df.show()         df.count()        df.collect()
```

**Quick heuristics for picking the right home:**

1. **SQL built-in?** → `F.*`. If writing `UPPER(x)`, `LENGTH(x)`, `YEAR(x)` in SQL, it's in `F`.
2. **About *this* column's identity/type/nullability?** → column method (`cast`, `alias`, `isNull`, `isin`, `between`, `desc`, `asc`).
3. **Per-row string test?** → column method (`contains`, `startswith`, `like`) — **not** `F`. This is the trap most people fall into.
4. **Operator (`+`, `>`, `&`, `~`)?** → inline; never `F`.
5. **Changes the shape of the data (rows/columns)?** → DataFrame method.

Important to lock in: when in doubt, **IDE autocomplete** is the fastest tiebreaker — type `df.col.` and see what methods appear; type `F.` and see what functions appear.

### Common DataFrame transformations

The ones that come up again and again:

**`.filter()`** — keep only rows satisfying the condition (analogous to SQL's `WHERE`). Spark provides `.where()` as a straight alias — use whichever reads better.

```python
# Only rows where vote_date is after 1 Jan 2019
df.filter(df.vote_date > '2019-01-01')

# Equivalent
df.where(df.vote_date > '2019-01-01')
```

**`.select()`** — return only the requested columns.

```python
df.select('name', 'position', 'vote_date')
```

**`.withColumn()`** — create a **new column**. First argument is the name; second is the expression that produces its values.

```python
import pyspark.sql.functions as F

# Add a 'year' column extracted from the vote_date column
df = df.withColumn('year', F.year(df.vote_date))
```

**`.drop()`** — remove a column from the DataFrame.

```python
df = df.drop('old_column')
```

Important to lock in: every one of these returns a **new DataFrame** — the original is unchanged (immutability from the previous chapter).

### Filtering data

Among the most common cleaning operations. `.filter()` is a workhorse for:

- **Removing null values**
- **Removing odd entries** — anything that doesn't fit the expected format
- **Splitting a DataFrame** containing combined data (e.g. one syslog file carrying multiple types of record)

Useful idioms:

```python
import pyspark.sql.functions as F

# Remove nulls in a specific column
df.filter(df.name.isNotNull())

# Keep only rows where 'name' contains 'Smith'
df.filter(df.name.contains('Smith'))

# Negate with ~ — exclude rows that contain 'Smith'
df.filter(~df.name.contains('Smith'))
```

Key point: the **tilde `~`** is the boolean NOT operator in PySpark expressions — works on any boolean column/condition, not just `.contains()`.

### Column string transformations

Modifying and converting strings is a big part of data cleaning. Most of the functions live in `pyspark.sql.functions`:

**Upper / lower casing (both via `F`):**

```python
# Create an 'upper' column with uppercase versions of 'name'
df = df.withColumn('upper', F.upper(df.name))

# Or lowercase
df = df.withColumn('lower', F.lower(df.name))
```

**Splitting strings → ArrayType (via `F`):**

```python
# Split 'name' on spaces → returns a column of lists (ArrayType)
df = df.withColumn('splits', F.split(df.name, ' '))
```

This is an example of an **intermediary column** — a column created purely as a stepping stone for further processing. Useful for keeping complex multi-step transformations readable.

**Type casting with `.cast()` (column method — not `F`):**

```python
from pyspark.sql.types import IntegerType

# Convert a string column to IntegerType
df = df.withColumn('age_int', df.age.cast(IntegerType()))
```

Important to remember: `cast()` is called on the **column object**, not on `F`. Common targets: `IntegerType()`, `FloatType()`, `DateType()`, etc.

### `ArrayType()` column functions

`ArrayType()` columns are the Spark equivalent of Python lists — a column where each row holds a list of values (often the result of `F.split(...)`).

Two go-to functions — **one in `F`, one a column method**:

**`F.size()`** — number of elements in the array column:

```python
# How many words after splitting on space
df = df.withColumn('word_count', F.size(df.splits))
```

**`.getItem(index)`** — grab the element at a specific index (0-based):

```python
# First word → 'first_name'
df = df.withColumn('first_name', df.splits.getItem(0))

# Second word → 'last_name'
df = df.withColumn('last_name',  df.splits.getItem(1))
```

Important to lock in: `getItem` is called on the **column itself**, not via `F`. Classic example of the F-vs-method split.

Spark has **many more** transformations and utility functions — worth consulting the official docs when building production pipelines.

### Key takeaways — cheat sheet

**The three homes at a glance:**

| Family | Pattern | Examples |
|---|---|---|
| **`F.*`** | `F.function(col)` | `F.upper`, `F.year`, `F.split`, `F.size`, `F.when`, `F.sum`, `F.lit` |
| **Column methods** | `col.method()` | `.cast`, `.alias`, `.isNull`, `.contains`, `.like`, `.isin`, `.between`, `.getItem`, `.desc` |
| **DataFrame methods** | `df.method()` | `.filter`, `.select`, `.withColumn`, `.drop`, `.groupBy`, `.orderBy`, `.join` |

**Transformations most used in cleaning:**

| Operation | What it does | Example |
|---|---|---|
| `.filter(cond)` / `.where(cond)` | Keep rows matching condition | `df.filter(df.age > 18)` |
| `.select(cols)` | Keep specified columns | `df.select('name', 'age')` |
| `.withColumn(name, expr)` | Add or replace a column | `df.withColumn('upper', F.upper(df.name))` |
| `.drop(col)` | Remove a column | `df.drop('temp_col')` |
| `.cast(type)` | Convert a column's type | `df.age.cast(IntegerType())` |

**String-manipulation one-liners (with `import pyspark.sql.functions as F`):**

```python
F.upper(col)           # UPPERCASE
F.lower(col)           # lowercase
F.split(col, sep)      # String → ArrayType
F.size(array_col)      # Length of an ArrayType column
array_col.getItem(i)   # i-th element of an ArrayType column
col.cast(IntegerType()) # type conversion
```

**Things to lock in:**

- DataFrames are **immutable** — reassign the result, don't expect in-place changes
- Conventional import: **`import pyspark.sql.functions as F`**
- **`F.*`** = SQL-style value transformations; **column methods** = questions/metadata about a column; **DataFrame methods** = shape-changing operations
- `.filter()` and `.where()` are **identical** — use whichever reads better
- **`~`** negates a boolean column/condition
- `.cast(...)` and `.getItem(i)` live on the **column**, not on `F` — classic gotchas
- Operators (`+`, `>`, `&`, `~`) are always inline, never via `F`
- When stuck on exact function names, reach for **IDE autocomplete** or the **PySpark SQL functions reference**

In [10]:
voter_df = spark.read.csv("DallasCouncilVoters.csv.gz", header = True)

voter_df.show()

+----------+-------------+-------------------+
|      DATE|        TITLE|         VOTER_NAME|
+----------+-------------+-------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|
|02/08/2017|Councilmember| Philip T. Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|
|02/08/2017|Councilmember|       Adam Medrano|
|02/08/2017|Councilmember|       Casey Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|
|02/08/2017|Councilmember|       Scott Griggs|
|02/08/2017|Councilmember|   B. Adam  McGough|
|02/08/2017|Councilmember|       Lee Kleinman|
|02/08/2017|Councilmember|      Sandy Greyson|
|02/08/2017|Councilmember|  Jennifer S. Gates|
|02/08/2017|Councilmember| Philip T. Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|
|02/08/2017|Councilmember|       Adam Medrano|
|02/08/2017|Councilmember|       Casey Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|
|02/08/2017|Councilmember| Rickey D. Callahan|
|01/11/2017|Councilmember|  Jennifer S. Gates|
|04/25/2018|C

In [11]:
# Show the distinct VOTER_NAME entries
voter_df.select('VOTER_NAME').distinct().show(40, truncate=False)

# Filter voter_df where the VOTER_NAME is 1-20 characters in length
voter_df = voter_df.filter('length(VOTER_NAME) > 0 and length(VOTER_NAME) < 20')

# Filter out voter_df where the VOTER_NAME contains an underscore
voter_df = voter_df.filter(~ F.col('VOTER_NAME').contains("_"))

# Show the distinct VOTER_NAME entries again
voter_df.select('VOTER_NAME').distinct().show(40, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|VOTER_NAME                                                                                                                                                                                                                                                                                                                                                                                                                 |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
# Add a new column called splits separated on whitespace
voter_df = voter_df.withColumn("splits", F.split(voter_df.VOTER_NAME, '\s+'))

# Create a new column called first_name based on the first item in splits
voter_df = voter_df.withColumn("first_name", voter_df.splits.getItem(0))

# Get the last entry of the splits list and create a column called last_name
voter_df = voter_df.withColumn("last_name", voter_df.splits.getItem(F.size('splits') - 1))

# Drop the splits column
voter_df = voter_df.drop('splits')

# Show the voter_df DataFrame
voter_df.show()

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2099/4199570017.py:2: SyntaxWarning: invalid escape sequence '\s'
  voter_df = voter_df.withColumn("splits", F.split(voter_df.VOTER_NAME, '\s+'))
/home/mothma/dc-pyspark/.venv/lib/python3.12/site-packages/pyspark/sql/column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


+----------+-------------+-------------------+----------+---------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_name|
+----------+-------------+-------------------+----------+---------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip| Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael| Rawlings|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|  Medrano|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|   Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|   Carolyn|   Arnold|
|02/08/2017|Councilmember|       Scott Griggs|     Scott|   Griggs|
|02/08/2017|Councilmember|   B. Adam  McGough|        B.|  McGough|
|02/08/2017|Councilmember|       Lee Kleinman|       Lee| Kleinman|
|02/08/2017|Councilmember|      Sandy Greyson|     Sandy|  Greyson|
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|
|02/08/2017|Councilmember| Philip T. Kingston|  

## Conditional DataFrame Column Operations


- Moves beyond **blanket transformations** to transformations that depend on the **content of each row**
- Covers: Spark's built-in conditional clauses **`.when()`** and **`.otherwise()`**, plus why to use them over hand-rolled Python if/else

### Conditional clauses — the idea

Everything covered so far has been a **blanket transformation** — applied to every row the same way, regardless of data.

Often what's actually needed is: "do X if the row looks like Y, else do Z". Traditional if / then / else, but **per row**.

Spark provides built-in conditional clauses for exactly this, and they're preferable to writing a Python if/else that runs per row:

- **Hand-rolled if/else** → each row evaluated independently in the Python interpreter → **serious performance degradation**, especially on large datasets (the UDF overhead kills the pipeline)
- **Built-in `.when()` / `.otherwise()`** → runs inside Spark's **optimised execution engine** → columnar, pushed down, catalyst-optimised

Two components to the conditional clauses:

- **`.when()`** — the "if ... then" part
- **`.otherwise()`** — the optional "else" part

### `.when()` — the if/then

Lives in `pyspark.sql.functions`. Takes **two arguments**:

1. The **condition** to check
2. The **value** to return if the condition is `True`

```python
import pyspark.sql.functions as F

# DataFrame with Name and Age columns
df.select(
    df.Name,
    df.Age,
    F.when(df.Age >= 18, 'Adult')
).show()
```

Result: each row with `Age >= 18` gets an extra column with `'Adult'`; otherwise the new column is **null** (nothing matched, no `.otherwise()` given).

Key points:

- The third argument to `.select()` is a **brand-new column** generated on the fly — no `.withColumn()` needed
- The new column is **unnamed by default** — Spark auto-labels it something like `CASE WHEN ... END`
- If the `.when()` condition fails and there's no `.otherwise()`, the cell is **null**

**Naming the generated column:**

Add `.alias()` to give it a sensible header:

```python
df.select(
    df.Name,
    df.Age,
    F.when(df.Age >= 18, 'Adult').alias('AgeGroup')
).show()
```

### Chaining `.when()` — the "else if" pattern

Multiple `.when()` calls can be chained for an if / else-if / else-if structure:

```python
df.select(
    df.Name,
    df.Age,
    F.when(df.Age >= 18, 'Adult')
     .when(df.Age <  18, 'Minor')
     .alias('AgeGroup')
).show()
```

Important to lock in:

- `.when()` can be chained **as many times as needed**
- Each `.when()` is evaluated **in order** — first match wins
- Rows matching **no** `.when()` clause still return **null** unless followed by `.otherwise()`

### `.otherwise()` — the else

Analogous to the `else` branch in traditional if/else. Takes a **single argument**: what to return when none of the preceding `.when()` clauses matched.

```python
df.select(
    df.Name,
    df.Age,
    F.when(df.Age >= 18, 'Adult')
     .otherwise('Minor')
     .alias('AgeGroup')
).show()
```

Result: every row gets either `'Adult'` or `'Minor'` — no nulls.

Important to remember:

- **Many `.when()` clauses** can be chained → but there can be **only one `.otherwise()`** per chain
- `.otherwise()` sits at the end — it's the fallback for everything that didn't match

Compare the two equivalent approaches:

```python
# With otherwise()
F.when(df.Age >= 18, 'Adult').otherwise('Minor')

# Without otherwise() — same result because the second .when() covers the rest
F.when(df.Age >= 18, 'Adult').when(df.Age < 18, 'Minor')
```

Both produce the same DataFrame. The `.otherwise()` form is usually cleaner when there's a clear default.

### Key takeaways — cheat sheet

**Pattern template:**

```python
import pyspark.sql.functions as F

df.withColumn(
    'new_col',
    F.when(cond1, value1)
     .when(cond2, value2)
     .otherwise(default_value)
)
```

**Side-by-side with traditional if/else:**

| Traditional code | Spark equivalent |
|---|---|
| `if cond: return x` | `F.when(cond, x)` |
| `elif cond2: return y` | `.when(cond2, y)` |
| `else: return z` | `.otherwise(z)` |

**Things to lock in:**

- Use **`F.when()` / `.otherwise()`** instead of hand-rolled Python if/else on each row — vastly better performance
- `.when()` takes **(condition, value)**; `.otherwise()` takes just a **value**
- Chain as many `.when()` clauses as needed; **only one `.otherwise()`** at the end
- First match wins — order the clauses from most-specific to most-general
- A row that matches **no** `.when()` and has no `.otherwise()` → **null**
- The generated column is unnamed by default — add **`.alias('name')`** or wrap in `.withColumn('name', ...)` to label it
- `.when()` lives in **`pyspark.sql.functions`** (the `F` family — a value-producing SQL-style function)

In [13]:
# Add a column to voter_df named random_val with the results of the F.rand() method for any voter with the title Councilmember
voter_df = voter_df.withColumn('random_val',
                               F.when(voter_df.TITLE == "Councilmember", F.rand()))

# Show some of the DataFrame rows, noting whether the when clause worked
voter_df.show()

+----------+-------------+-------------------+----------+---------+--------------------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_name|          random_val|
+----------+-------------+-------------------+----------+---------+--------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|0.016826943124080396|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip| Kingston|  0.7272829157366502|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael| Rawlings|                NULL|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|  Medrano|   0.578001706908305|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|   Thomas| 0.04668079973931327|
|02/08/2017|Councilmember|Carolyn King Arnold|   Carolyn|   Arnold|  0.5762536652283116|
|02/08/2017|Councilmember|       Scott Griggs|     Scott|   Griggs|  0.7510226433054144|
|02/08/2017|Councilmember|   B. Adam  McGough|        B.|  McGough|  0.6224722373169422|
|02/08/2017|Councilme

In [14]:
# Add a column to `voter_df` named `random_val` with the results of the `F.rand()` method for any voter with the title Councilmember. Set `random_val` to 2 for the Mayor. Set any other title to the value 0.
voter_df = voter_df.withColumn('random_val',
                               F.when(voter_df.TITLE == 'Councilmember', F.rand())
                               .when(voter_df.TITLE == 'Mayor', 2)
                               .otherwise(0)
)

# Show some of the Data Frame rows, noting whether the clauses worked.
voter_df.show()

# Use the `.filter` clause to find 0 in `random_val`.
voter_df.filter(voter_df.random_val ==0).show()

+----------+-------------+-------------------+----------+---------+--------------------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_name|          random_val|
+----------+-------------+-------------------+----------+---------+--------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|  0.6696540353975636|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip| Kingston|  0.5754973034198724|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael| Rawlings|                 2.0|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|  Medrano|  0.5472605474289425|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|   Thomas|  0.6998780005820834|
|02/08/2017|Councilmember|Carolyn King Arnold|   Carolyn|   Arnold|0.041691072338955415|
|02/08/2017|Councilmember|       Scott Griggs|     Scott|   Griggs|  0.9544100405089215|
|02/08/2017|Councilmember|   B. Adam  McGough|        B.|  McGough|  0.7244021493178178|
|02/08/2017|Councilme

## User Defined Functions

- Built-in Spark functions cover most cases — but sometimes **custom logic** is needed
- A **UDF** = a Python function wrapped so Spark can apply it to a column
- Covers: how to define, how to register with `F.udf()`, the `@udf` decorator shortcut, how to call, plus the performance caveat worth knowing up front

### What is a UDF?

**UDF** (User Defined Function) = a plain Python method the user writes to perform some specific bit of logic that no built-in Spark function covers.

**The workflow:**

1. Write a normal Python function
2. Wrap it with `pyspark.sql.functions.udf()`, telling Spark what **return type** to expect
3. Store the wrapped version as a variable
4. Use it like any other Spark function (inside `withColumn`, `select`, etc.)

Under the hood, Spark calls the Python function **once per row**, passing the value of the specified column(s) in and receiving the return value back.

### Example — a `reverseString` UDF

Trivial-but-illustrative example: reverse the characters of a string.

**Step 1 — define the Python function:**

```python
def reverseString(mystr):
    return mystr[::-1]   # Python slice shorthand for reverse
```

Nothing Spark-specific — just regular Python. Given `"help"`, returns `"pleh"`.

**Step 2 — wrap it with `F.udf()`:**

```python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

udfReverseString = F.udf(reverseString, StringType())
```

Two arguments to `F.udf()`:

- The **Python function** itself
- The **Spark return type** (any type from `pyspark.sql.types` — can even be a complex `ArrayType` or a full schema)

Typical return types: `StringType()`, `IntegerType()`, `FloatType()`, `ArrayType(...)`.

**Step 3 — apply it to a DataFrame:**

```python
user_df = user_df.withColumn(
    'ReverseName',
    udfReverseString(user_df.Name)
)
```

The column `user_df.Name` is passed to the UDF; Spark calls the Python function row-by-row and stashes the results in the new `ReverseName` column.

### Syntactic sugar — the `@udf` decorator

Python decorators give a cleaner one-step way to write the exact same thing:

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(StringType())
def reverseString(mystr):
    return mystr[::-1]

user_df = user_df.withColumn('ReverseName', reverseString(user_df.Name))
```

**How it relates to the function-call form:**

Decorators are just sugar — these two are **exactly equivalent** under the hood:

```python
@udf(StringType())
def reverseString(mystr):
    return mystr[::-1]

# ↑↑↑ expands to ↓↓↓

def reverseString(mystr):
    return mystr[::-1]
reverseString = udf(StringType())(reverseString)
```

Same wrapping, same behaviour, **same performance characteristics**. Pick whichever reads better.

**Decorator shortcuts worth knowing:**

```python
@udf                     # no parens — defaults return type to StringType()
def greet(name):
    return f"Hello, {name}"

@udf("int")              # Spark SQL type string also works
def length(s):
    return len(s)

@udf("array<string>")    # complex type via SQL string
def split_comma(s):
    return s.split(",")
```

**When to reach for which:**

| Situation | Favour |
|---|---|
| Python function also used *outside* Spark (tests, other code paths) | `F.udf(fn, type)` — keeps original `fn` name intact |
| Python function exists **only** to be called through Spark | `@udf(type)` — one block, type sits next to definition |
| Teaching material / clarity for a beginner audience | `F.udf(fn, type)` — more explicit about what's happening |

Important to remember: the decorator doesn't buy any speed — the Python UDF overhead applies equally to both forms. It's **purely a syntax preference**.

### Argument-less UDFs

UDFs don't have to take a column argument — they can produce a value on their own.

Example: pick a random "house" for each row.

```python
import random
from pyspark.sql.types import StringType

def sortingCap():
    return random.choice(['G', 'H', 'R', 'S'])

udfSortingCap = F.udf(sortingCap, StringType())

user_df = user_df.withColumn('House', udfSortingCap())
```

Equivalent with the decorator:

```python
@udf(StringType())
def sortingCap():
    return random.choice(['G', 'H', 'R', 'S'])

user_df = user_df.withColumn('House', sortingCap())
```

Key differences from the previous example:

- No column passed in — just `udfSortingCap()` / `sortingCap()` with empty parentheses
- Still called **once per row**, so each row gets its own random letter

### Important caveat — performance

UDFs are a **big escape hatch**, but they come at a real cost. Important to lock in:

- The Python function runs in a separate **Python interpreter** process on each executor
- For every row, Spark has to serialise the value **from JVM to Python**, run the function, then serialise the result **back from Python to JVM**
- **Catalyst cannot optimise inside the UDF** — it's a black box to the query planner
- No **vectorisation**, no **predicate pushdown** through the UDF

So the rule of thumb:

- If a **built-in function** from `F` can do it → **use that**, every time
- If `F.when() / .otherwise()` can express the logic → **use that**
- Reach for a UDF only when **no combination of built-ins works**

For serious performance, Spark also offers **pandas UDFs** (vectorised UDFs using Apache Arrow), which batch many rows into a pandas Series and process them in one Python call — orders of magnitude faster than row-at-a-time UDFs. Worth knowing about for later.

### Key takeaways — cheat sheet

**Two equivalent UDF templates:**

```python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Function-call form
def my_fn(x):
    return ...

udf_my_fn = F.udf(my_fn, StringType())
df = df.withColumn('new_col', udf_my_fn(df.some_col))

# Decorator form — syntactic sugar for the above
@F.udf(StringType())
def my_fn(x):
    return ...

df = df.withColumn('new_col', my_fn(df.some_col))
```

**Things to lock in:**

- **UDF** = plain Python function + `F.udf(fn, return_type)` wrapper + applied like any Spark function
- Second argument to `F.udf()` is the **return type** (from `pyspark.sql.types`)
- **`@udf(ReturnType())`** is pure syntactic sugar for the same wrapping — no speed difference, just prettier
- `@udf` with no parens defaults to **`StringType()`** — convenient, but always better to specify the type explicitly
- Spark SQL type strings (`"int"`, `"string"`, `"array<string>"`) work in place of the typed classes
- Called **once per row** — Python runs for every row
- Argument-less UDFs are fine — call with empty parens
- **Performance cost is real** — JVM↔Python serialisation per row, no Catalyst optimisation
- Prefer **built-in `F` functions** whenever they can do the job
- For heavy workloads, **pandas UDFs** (vectorised) are the better escape hatch

In [15]:
voter_df = voter_df.withColumn("splits", F.split(voter_df.VOTER_NAME, '\s+'))

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2099/1354151728.py:1: SyntaxWarning: invalid escape sequence '\s'
  voter_df = voter_df.withColumn("splits", F.split(voter_df.VOTER_NAME, '\s+'))


In [16]:
def getFirstAndMiddle(names):
  # Edit the `getFirstAndMiddle()` function to return a space separated string of names, except the last entry in the names list.
  return ' '.join(names[:-1])

# Define the function as a user-defined function. It should return a string type.
udfFirstAndMiddle = F.udf(getFirstAndMiddle, StringType())

# Create a new column on `voter_df` called `first_and_middle_name` using your UDF.
voter_df = voter_df.withColumn('first_and_middle_name', udfFirstAndMiddle(voter_df.splits))

# Show the Data Frame.
voter_df.show()

+----------+-------------+-------------------+----------+---------+--------------------+--------------------+---------------------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_name|          random_val|              splits|first_and_middle_name|
+----------+-------------+-------------------+----------+---------+--------------------+--------------------+---------------------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|    Gates|  0.6696540353975636|[Jennifer, S., Ga...|          Jennifer S.|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip| Kingston|  0.5754973034198724|[Philip, T., King...|            Philip T.|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael| Rawlings|                 2.0|[Michael, S., Raw...|           Michael S.|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|  Medrano|  0.5472605474289425|     [Adam, Medrano]|                 Adam|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|   Thomas|  0.69987

## Partitioning and Lazy Processing

- Steps back from transformations to look at **how Spark actually processes data** under the hood
- Covers: **partitioning**, **lazy processing** (recap with a new angle), the problem of adding **IDs** in a distributed world, and `monotonically_increasing_id()`

### Partitioning

Spark breaks DataFrames into **partitions** — chunks of data distributed across the cluster.

- Partitions can be automatically **defined, enlarged, or shrunk**
- Count and size vary with the cluster type and configuration
- **Rule of thumb**: try to keep partition sizes roughly equal (skewed partitions kill parallelism — one slow worker holds up everything)

**Why partitions matter for performance:**

Each partition is handled **independently** by whichever node picks it up. That independence is the key to Spark's horizontal scaling:

- Workers don't need to **compete for shared resources**
- Workers don't need to **consult other nodes** for answers
- Each worker can schedule its own work optimally

Important to lock in: this independence is *why* Spark scales. Any operation that forces workers to coordinate (shuffles, global sequences, etc.) is expensive precisely because it breaks the independence.

### Lazy processing (revisited)

Previously covered — quick refresher with a new angle:

- Any **transformation** is lazy — it's more like a **recipe** than a command
- Transformations: `.withColumn()`, `.select()`, `.filter()`, `.drop()`, etc. → they **define** what to do, don't actually do it
- **Actions** (`.count()`, `.show()`, `.write()`, `.collect()`) trigger execution

The new wrinkle worth noting:

- **Spark can reorder transformations** for best performance
- Usually invisible — results are the same
- **Occasionally causes unexpected behaviour** — e.g. IDs not being assigned until after other transformations run
- Doesn't produce *wrong* answers, but the data can look **unusual** if the reorder isn't anticipated

This is the kind of thing that catches people out when they add `monotonically_increasing_id()` expecting sequential numbers and then wonder why joins look odd.

### Adding IDs — the problem

Relational databases typically have an **ID column** — an integer, sequential, unique — used to identify rows (foreign keys, joins, deduplication).

**Why traditional sequential IDs don't fit Spark:**

- Each new ID = previous + 1
- Multiple workers would all need to consult a **shared counter** for "what's next"
- Fine on a single server
- A **serious bottleneck** on a distributed platform — forces coordination between workers that otherwise wouldn't talk to each other

So Spark can't just generate sequential IDs the way Postgres does. A different design is needed.

### `monotonically_increasing_id()`

Spark's built-in solution. Produces an ID that is:

- **An integer**
- **Increasing in value**
- **Unique**
- **Parallel-friendly** — each partition gets its own range, no coordination needed

**Crucial catch:** the IDs are **not sequential**. There can be large gaps between values.

```python
import pyspark.sql.functions as F

df = df.withColumn('row_id', F.monotonically_increasing_id())
df.show()
```

Sample output (illustrative):

```
+------+----+-------------+
|  name| age|       row_id|
+------+----+-------------+
|Alice |  30|            0|
|Bob   |  25|            1|
|Carol |  40|  8589934592 |
|David |  35|  8589934593 |
+------+----+-------------+
```

Notice: integers, increasing, unique — but **not consecutive**. The jump between rows 2 and 3 is where a new partition started.

**How it works under the hood:**

- The ID is a **64-bit integer**, effectively split into groups per partition
- Each partition gets up to **~8.4 billion IDs** to assign within its own range
- There are **~2.1 billion possible partitions**, and the ranges **never overlap**
- That's how uniqueness holds without any inter-worker coordination

Key insight: the ID's **top bits encode the partition number**; the **bottom bits** are the sequential position within that partition. Because each partition owns a disjoint range, no worker ever has to ask another one what to use next.

### Practical notes

A few practical points worth remembering:

- Partitioning and `monotonically_increasing_id()` have **a lot of nuance** — the above is the shape, not the whole story
- Spark being **lazy** is often the key to troubleshooting weird behaviour — if something looks out of order, remember: the transformations may have been **reordered** before execution
- **Operations can be out of order**, especially when **joins** are involved — a join can re-partition data, scrambling what felt like a stable ordering
- Always **test transformations** on small data before trusting assumptions about order

### Key takeaways — cheat sheet

**Partition + ID model:**

| Concept | Key point |
|---|---|
| **Partition** | Independent chunk of a DataFrame; workers process partitions without coordinating |
| **Lazy transformations** | Can be reordered by Spark's optimiser before execution |
| **Traditional sequential IDs** | Require shared state → bottleneck for distributed systems |
| **`monotonically_increasing_id()`** | Unique, increasing, **not sequential**; each partition owns its own range |

**ID generation template:**

```python
import pyspark.sql.functions as F

df_with_id = df.withColumn('row_id', F.monotonically_increasing_id())
```

**Things to lock in:**

- Spark's parallelism comes from **partitions acting independently** — any operation forcing coordination costs performance
- Keep partition sizes roughly **balanced** when possible — skew is a silent killer
- Transformations are **lazy and reorderable**; don't assume execution order from code order
- `monotonically_increasing_id()` gives **unique, increasing** IDs — **not consecutive**
- Each partition owns its own ID range (~8.4 billion values), so no worker coordination needed → no bottleneck
- If joins or other partition-shuffling operations are in the chain, IDs may appear in unexpected places — **test on small data first**

In [17]:
df = spark.read.format("csv").options(header=True).load("DallasCouncilVotes.csv.gz")

df.printSchema()

root
 |-- DATE: string (nullable = true)
 |-- AGENDA_ITEM_NUMBER: string (nullable = true)
 |-- ITEM_TYPE: string (nullable = true)
 |-- DISTRICT: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER NAME: string (nullable = true)
 |-- VOTE CAST: string (nullable = true)
 |-- FINAL ACTION TAKEN: string (nullable = true)
 |-- AGENDA ITEM DESCRIPTION: string (nullable = true)
 |-- AGENDA_ID: string (nullable = true)
 |-- VOTE_ID: string (nullable = true)



In [18]:
# Select the unique entries from the column `VOTER NAME` and create a new DataFrame called `voter_df`.
voter_df = df.select(df["VOTER NAME"]).distinct()

# # Count the rows in the `voter_df` DataFrame.
print("\nThere are %d rows in the voter_df DataFrame.\n" % voter_df.count())

# # Add a ROW_ID column using the appropriate Spark function.
voter_df = voter_df.withColumn('ROW_ID', F.monotonically_increasing_id())

# Show the rows with the 10 highest ROW_IDs.
voter_df.orderBy(voter_df.ROW_ID.desc()).show(10)


There are 36 rows in the voter_df DataFrame.

+--------------------+------+
|          VOTER NAME|ROW_ID|
+--------------------+------+
|                NULL|    35|
|        Lee Kleinman|    34|
|  the  final  201...|    33|
|         Erik Wilson|    32|
|  the  final   20...|    31|
| Carolyn King Arnold|    30|
| Rickey D.  Callahan|    29|
|   the   final  2...|    28|
|    Monica R. Alonzo|    27|
|     Lee M. Kleinman|    26|
+--------------------+------+
only showing top 10 rows



In [19]:
voter_df_single = voter_df.select(F.col("VOTER NAME")).repartition(1)
voter_df = voter_df.select(F.col("VOTER NAME")).repartition(2)

In [20]:
# Print the number of partitions in each DataFrame
print("\nThere are %d partitions in the voter_df DataFrame.\n" % voter_df.rdd.getNumPartitions())
print("\nThere are %d partitions in the voter_df_single DataFrame.\n" % voter_df_single.rdd.getNumPartitions())

# Add a ROW_ID field to each DataFrame
voter_df = voter_df.withColumn('ROW_ID', F.monotonically_increasing_id())
voter_df_single = voter_df_single.withColumn('ROW_ID', F.monotonically_increasing_id())

# Show the top 10 IDs in each DataFrame 
voter_df.orderBy(voter_df.ROW_ID.desc()).show(10)
voter_df_single.orderBy(voter_df_single.ROW_ID.desc()).show(10)


There are 2 partitions in the voter_df DataFrame.


There are 1 partitions in the voter_df_single DataFrame.

+--------------------+----------+
|          VOTER NAME|    ROW_ID|
+--------------------+----------+
|        Omar Narvaez|8589934609|
| Carolyn King Arnold|8589934608|
|       Mark  Clayton|8589934607|
|    Monica R. Alonzo|8589934606|
|         Erik Wilson|8589934605|
|       Scott  Griggs|8589934604|
|   the   final  2...|8589934603|
|  Philip T. Kingston|8589934602|
| the final 2018 A...|8589934601|
|   Tiffinni A. Young|8589934600|
+--------------------+----------+
only showing top 10 rows

+--------------------+------+
|          VOTER NAME|ROW_ID|
+--------------------+------+
|                NULL|    35|
|        Lee Kleinman|    34|
|  the  final  201...|    33|
|         Erik Wilson|    32|
|  the  final   20...|    31|
| Carolyn King Arnold|    30|
| Rickey D.  Callahan|    29|
|   the   final  2...|    28|
|    Monica R. Alonzo|    27|
|     Lee M. Kleinman|    2

In [21]:
voter_df = spark.read.format("csv").options(header=True, inferSchema=True).load("DallasCouncilVotes.csv.gz")

voter_df_march = (
    voter_df
        .filter(F.col("DATE").startswith("03/"))
        .select(F.col("VOTER NAME"))
        .withColumn("ROW_ID", F.monotonically_increasing_id())
).repartition(1)
voter_df_april = voter_df.filter(F.col("DATE").startswith("04/")).select(F.col("VOTER NAME")).repartition(1)

In [22]:
# Determine the highest `ROW_ID` in `voter_df_march` and save it in the variable `previous_max_ID`.
previous_max_ID = voter_df_march.select('ROW_ID').rdd.max()[0]

# Add a `ROW_ID` column to `voter_df_april` starting at the value of `previous_max_ID` + 1.
voter_df_april = voter_df_april.withColumn('ROW_ID', F.monotonically_increasing_id() + previous_max_ID + 1)

# Show the `ROW_ID`'s from both DataFrames and compare
voter_df_april.select('ROW_ID').show()
voter_df_march.select('ROW_ID').show()

+------+
|ROW_ID|
+------+
|  1110|
|  1111|
|  1112|
|  1113|
|  1114|
|  1115|
|  1116|
|  1117|
|  1118|
|  1119|
|  1120|
|  1121|
|  1122|
|  1123|
|  1124|
|  1125|
|  1126|
|  1127|
|  1128|
|  1129|
+------+
only showing top 20 rows

+------+
|ROW_ID|
+------+
|     0|
|     1|
|     2|
|     3|
|     4|
|     5|
|     6|
|     7|
|     8|
|     9|
|    10|
|    11|
|    12|
|    13|
|    14|
|    15|
|    16|
|    17|
|    18|
|    19|
+------+
only showing top 20 rows



# III

## Caching


- Improving the performance of Spark data cleaning tasks using caching

### What is caching?

- Caching = storing the results of a DataFrame **in memory or on disk** on the processing nodes of the cluster
- Speeds up subsequent transformations/actions, since the data most likely no longer has to be fetched from the original source
- Reduces resource utilisation across the cluster: less storage access, networking and CPU on the Spark nodes, because the data is probably already present

### Disadvantages of caching

- Very large datasets may not fit in the memory reserved for cached DataFrames
- Depending on the later transformations, the cache may do nothing for performance
- If the data does not stay cached in memory, it may be persisted to disk
    - Depending on the cluster's disk configuration, this may not be much of an improvement
    - Reading from a local network resource + slow local disk I/O → may be better to avoid caching altogether
- The lifetime of a cached object is **not guaranteed**
    - Spark regenerates the DataFrame automatically if the cache is lost, but this can cause processing delays

### Caching tips

- Only cache if the DataFrame is going to be **used again**; not worth it for a single task
- Manually stop caching a DataFrame once finished with it; this frees cache resources for other DataFrames

#### Testing configurations

- Best way to gauge the benefit of caching is to test:
    - Cache at various points in the processing cycle
    - Check whether processing time actually improves

```python
import time

start_time = time.time()

# First count: reads from source and populates the cache
voter_df = spark.read.csv('voter_data.txt.gz', header=True).cache()
print("Rows: %d, took %f seconds" % (voter_df.count(), time.time() - start_time))

# Second count: served from the cache, should be much quicker
start_time = time.time()
print("Rows: %d, took %f seconds" % (voter_df.count(), time.time() - start_time))
```

#### Storage choice

- Cache in memory or on fast NVMe / SSD storage
    - SSD is slower than main memory but drastically faster than spinning disk
- Local spinning hard drives can still be useful when:
    - A large DataFrame takes a lot of steps to generate
    - The data has to be accessed over the Internet

#### Intermediate Parquet representations

- Worth trying if normal caching does not seem to help (same approach as Chapter 1)
- Acts as a checkpoint if a job fails mid-task
- Can still be combined with caching for further gains

```python
# Write out an intermediate representation part-way through the pipeline
voter_df.write.parquet('voter_data_intermediate.parquet', mode='overwrite')

# Read it back and cache it for the remaining steps
voter_df = spark.read.parquet('voter_data_intermediate.parquet').cache()
```

### Implementing caching

- Call `.cache()` on a DataFrame **before** an action
- Takes no arguments
- Important to remember: `.cache()` is itself a **transformation** (lazy), so nothing is cached until an action is called

#### Chaining .cache() before the action

```python
voter_df = spark.read.csv('voter_data.txt.gz')
voter_df.cache().count()
```

#### Calling .cache() separately

```python
from pyspark.sql.functions import monotonically_increasing_id

# Transformation: add an ID column
voter_df = voter_df.withColumn('ID', monotonically_increasing_id())

# Mark the DataFrame for caching
voter_df = voter_df.cache()

# Action: this is the point at which voter_df actually gets cached
voter_df.show()
```

### More cache operations

#### Checking cache status

- `.is_cached`: boolean **property** (no brackets), returns `True` or `False`

```python
print(voter_df.is_cached)   # True
```

#### Removing from the cache

- `.unpersist()`: removes the DataFrame from the cache, no arguments

```python
voter_df.unpersist()

print(voter_df.is_cached)   # False
```

### Key takeaways

- Caching stores DataFrame results in memory/on disk on the cluster nodes → faster repeat access, lower resource utilisation
- Not free: large data may not fit, disk-backed cache can be slow, cache lifetime is not guaranteed
- Only cache DataFrames that get reused, and always **test** whether it helps
- Prefer memory or NVMe/SSD; fall back to intermediate Parquet files if caching alone is not effective
- `.cache()` is lazy; an action (`.count()`, `.show()`, ...) triggers the actual caching

#### Cheat sheet

| Operation | Syntax | Notes |
|---|---|---|
| Cache a DataFrame | `df.cache()` | Transformation, no arguments |
| Trigger the cache | `df.count()` / `df.show()` | Any action |
| Check cache status | `df.is_cached` | Property, returns `True`/`False` |
| Remove from cache | `df.unpersist()` | No arguments, frees cache resources |
| Parquet checkpoint | `df.write.parquet(path)` → `spark.read.parquet(path)` | Useful if a job fails mid-task |

In [23]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("clean-data-w-spark").getOrCreate()

26/11/20 18:47:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [24]:
departures_df = spark.read.format('csv').options(header=True).load('AA_DFW_2017_Departures_Short.csv.gz')

departures_df.show()

+-----------------+-------------+-------------------+-----------------------------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|
+-----------------+-------------+-------------------+-----------------------------+
|       01/01/2017|         0005|                HNL|                          537|
|       01/01/2017|         0007|                OGG|                          498|
|       01/01/2017|         0037|                SFO|                          241|
|       01/01/2017|         0043|                DTW|                          134|
|       01/01/2017|         0051|                STL|                           88|
|       01/01/2017|         0060|                MIA|                          149|
|       01/01/2017|         0071|                LAX|                          203|
|       01/01/2017|         0074|                MEM|                           76|
|       01/01/2017|         0081|                DEN|                       

In [25]:
import time

start_time = time.time()

# Cache the unique rows in the `departures_df` DataFrame.
departures_df = departures_df.distinct().cache()

# Perform a count query on `departures_df`, noting how long the operation takes.
print("Counting %d rows took %f seconds" % (departures_df.count(), time.time() - start_time))

# Count the rows again, noting the variance in time of a cached DataFrame.
start_time = time.time()
print("Counting %d rows again took %f seconds" % (departures_df.count(), time.time() - start_time))

Counting 139358 rows took 2.743660 seconds
Counting 139358 rows again took 0.617572 seconds


In [26]:
# Check the caching status on the `departures_df` DataFrame.
print("Is departures_df cached?: %s" % departures_df.is_cached)
print("Removing departures_df from cache")

# Remove the `departures_df` DataFrame from the cache.
departures_df.unpersist()

# Validate the caching status again.
print("Is departures_df cached?: %s" % departures_df.is_cached)

Is departures_df cached?: True
Removing departures_df from cache
Is departures_df cached?: False


## Improve import performance


- Caching speeds up repeated work on a DataFrame; the next step is speeding up getting data *into* a DataFrame
- Covered here:
    - How Spark clusters are organised (driver and workers)
    - Why the number and size of import objects matters
    - How schemas affect import speed
    - Ways to split large objects into smaller ones

### Spark clusters

- A Spark cluster is made up of two types of processes:
    - **One driver process**
    - **As many worker processes as required**
- **Driver**
    - Handles task assignment
    - Consolidates the data results coming back from the workers
- **Workers**
    - Typically handle the actual transformation / action tasks of a Spark job
    - Once assigned tasks, operate fairly independently
    - Report results back to the driver
- A single node cluster is possible (used throughout this course) but rarely seen in production
- Several ways exist to run a Spark cluster; the method depends on the specific environment

```python
from pyspark.sql import SparkSession

# Single node "cluster": driver and workers all on the local machine
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("import_performance") \
    .getOrCreate()
```

### Import performance

- Important to understand how the cluster implements an import job
- The process varies by task type, but the general rule: **the more import objects available, the better the cluster can divide up the job**
    - Little difference on a single node cluster
    - On a larger cluster, each worker can take part in the import
- **One large file performs considerably worse than many smaller files**
    - Depending on cluster configuration, a very large file may not be processable at all, while the same amount of data split across smaller files is handled easily
- A single import statement can cover multiple files
    - Any standard wildcard symbol can be used in the import filename

```python
# One import statement, many files (airport-0001.txt.gz, airport-0002.txt.gz, ...)
airport_df = spark.read.csv("airport-*.txt.gz", header=True)

# Other wildcard forms work as well
departures_df = spark.read.csv("departures_201[5-8]_??.csv", header=True)
```

- Less important, but still helpful: **objects of roughly the same size** perform better than a mix of very large and very small objects

### Schemas

- Well-defined schemas drastically improve import performance
- Without a schema:
    - Import tasks must read the data multiple times to infer the structure
    - Very slow with a lot of data
    - Spark may not define the objects in the data the same way as intended (e.g. a numeric ID read as a string, or the reverse)
- Schemas also provide **validation on import**
    - Saves steps in data cleaning jobs
    - Improves overall processing time

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define the schema up front rather than relying on inferSchema=True
departures_schema = StructType([
    StructField("flight_date", StringType(), False),
    StructField("flight_number", IntegerType(), False),
    StructField("destination_airport", StringType(), True),
    StructField("flight_duration", IntegerType(), True)
])

departures_df = spark.read.csv(
    "departures-*.csv",
    header=True,
    schema=departures_schema
)

departures_df.printSchema()
```

### How to split objects

- Several effective ways to split an object (mostly files) into more, smaller objects

#### OS utilities

- Built-in tools such as `split`, `cut` or `awk`
- Example with `split`:

```bash
split -l 10000 -d largefile chunk-
```

- `-l 10000` → number of lines per output file
- `-d` → use numeric suffixes
- `largefile` → name of the file to be split
- `chunk-` → prefix for the output files
- A 10M-record `largefile` at 10,000 lines per file produces 1,000 numbered chunk files

#### Custom scripts

- Python (or any other language) can split the objects in whatever way is needed

```python
# Plain Python: split a large file into 10,000-line chunks
lines_per_file = 10000
chunk, file_number = [], 0

with open("largefile") as source:
    for line in source:
        chunk.append(line)
        if len(chunk) == lines_per_file:
            with open("chunk-%04d" % file_number, "w") as target:
                target.writelines(chunk)
            chunk, file_number = [], file_number + 1

# Write any remaining lines
if chunk:
    with open("chunk-%04d" % file_number, "w") as target:
        target.writelines(chunk)
```

#### Read once, write out as Parquet

- Useful when splitting tools are not available and the DataFrame will be used often
- Read in the single large file, then write it back out as Parquet
    - The initial import is slow, but later analysis works well
- Important to remember: if hitting limits because of cluster sizing, do **as little processing as possible** before writing to Parquet

```python
# Slow one-off import of the single large file
df_csv = spark.read.csv("singlelargefile.csv", header=True, schema=departures_schema)

# Write straight out to Parquet with minimal processing
df_csv.write.parquet("data.parquet", mode="overwrite")

# Use the Parquet version for all later work
df = spark.read.parquet("data.parquet")
```

### Key takeaways

- **Cluster** = 1 driver (assigns tasks, consolidates results) + many workers (run transformations / actions)
- **Many small files beat one large file**: more objects means more workers sharing the import
- **Similar-sized objects** perform better than a mix of very large and very small
- **Wildcards** allow one import statement for many files
- **Define schemas**: avoids multiple reads for inference, gives correct types, validates on import
- **Splitting options**: OS utilities, a custom script, or read once and write to Parquet
- When cluster size is the limit, keep processing minimal before the Parquet write

| Task | Syntax |
|---|---|
| Import many files at once | `spark.read.csv("airport-*.txt.gz", header=True)` |
| Import with a schema | `spark.read.csv(path, header=True, schema=my_schema)` |
| Split a file by lines | `split -l 10000 -d largefile chunk-` |
| Write to Parquet | `df.write.parquet("data.parquet", mode="overwrite")` |
| Read Parquet | `spark.read.parquet("data.parquet")` |

In [27]:
# # Import the `departures_full.txt.gz` file and the `departures_xxx.txt.gz` files into separate DataFrames.
# full_df = spark.read.csv('____')
# split_df = ____(____)

# # Run a count on each DataFrame and compare the run times.
# start_time_a = time.time()
# print("Total rows in full DataFrame:\t%d" % ____)
# print("Time to run: %f" % (time.time() - start_time_a))

# start_time_b = time.time()
# print("Total rows in split DataFrame:\t%d" % ____)
# print("Time to run: %f" % (time.time() - start_time_b))

## Cluster sizing tips


- Follows on from improving import performance
- Covered here:
    - Spark configuration options and how to read / write them
    - Cluster types
    - The driver: responsibilities and sizing
    - The worker: responsibilities and sizing

### Configuration options

- Spark has many configuration settings controlling all aspects of the installation
- Settings can be modified to best match the specific needs of the cluster
- Configurations are available in three places:
    - Configuration files
    - The Spark web interface
    - Run-time code
- The course test cluster is only accessible via command shell, so run-time code is the option used
- **Reading a setting**: `spark.conf.get()` with the name of the setting as the argument
- **Writing a setting**: `spark.conf.set()` with the name of the setting and the value as the arguments

```python
# Read configuration settings
app_name = spark.conf.get("spark.app.name")
driver_tcp_port = spark.conf.get("spark.driver.port")
num_partitions = spark.conf.get("spark.sql.shuffle.partitions")

print("Name: %s" % app_name)
print("Driver TCP port: %s" % driver_tcp_port)
print("Number of partitions: %s" % num_partitions)

# Write a configuration setting
spark.conf.set("spark.sql.shuffle.partitions", 500)

# Confirm the change
print(spark.conf.get("spark.sql.shuffle.partitions"))
```

### Cluster types

- Spark deployments vary depending on the exact needs of the users
- A large component of any deployment is the **cluster management mechanism**
- Spark clusters can be:
    - **Single node**: all components deployed on a single system (physical / VM / container)
    - **Standalone**: dedicated machines as the driver and workers
    - **Managed**: cluster components handled by a third party cluster manager such as YARN, Mesos or Kubernetes
- The course uses a single node cluster
- Production environments vary wildly; standalone clusters are the reference point because the concepts carry across all management types

```python
from pyspark.sql import SparkSession

# The master URL reflects the cluster type
# Single node:   "local[*]"
# Standalone:    "spark://driver-host:7077"
# Managed:       "yarn" or "k8s://https://k8s-api-host:6443"
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("cluster_sizing") \
    .getOrCreate()

# Check which type of cluster is in use
print(spark.conf.get("spark.master"))
```

### Driver

- One driver per Spark cluster
- Responsibilities:
    - **Task assignment** to the various nodes / processes in the cluster
    - **Monitoring** the state of all processes and tasks, and handling any task retries
    - **Consolidating results** from the other processes in the cluster
    - **Handling access to shared data** and verifying each worker process has the necessary resources (code, data, etc.)
- Sizing tips:
    - Given its importance, often worth giving the driver node a higher specification than the other systems
    - **Double the memory** of the other nodes is recommended; useful for task monitoring and data consolidation
    - **Fast local storage** is useful, as with all Spark systems

```python
# Inspect the driver's settings at run time
print(spark.conf.get("spark.driver.host"))
print(spark.conf.get("spark.driver.memory", "1g"))   # second argument = default if unset
```

```bash
# Driver memory is set at launch, e.g. double the executor memory
spark-submit --driver-memory 16g --executor-memory 8g departures_job.py
```

### Worker

- Runs the tasks assigned by the driver and communicates the results back to the driver
- Ideally has a copy of all code and data, plus access to the resources needed to complete a given task
    - If any of these are unavailable, the worker must pause to obtain them
- Sizing recommendations:
    - **More worker nodes is often better than larger nodes**, depending on the type of task
        - Especially obvious during import and export operations, as more machines are available to do the work
    - **Test various configurations** to find the right balance for the workload
        - Cloud example: 16 worker nodes may complete a job in 1 hour at a cost of $50
        - An 8 worker configuration might take 1.25 hours but cost only about half as much
    - **Fast local storage (SSD / NVMe)** is used for caching, intermediate files, etc.

```python
# Check how the work is being divided
print(spark.conf.get("spark.default.parallelism", "not set"))

departures_df = spark.read.csv("departures_*.txt.gz", header=True)

# Number of partitions = number of pieces the workers can process in parallel
print("Partition count: %d" % departures_df.rdd.getNumPartitions())
```

### Key takeaways
```
Cluster  = all the nodes together
  └─ Node = one machine
       ├─ runs the Driver   (the one coordinator)   ← only on one node
       └─ runs a Worker     (does the actual tasks) ← on the other nodes
```
- Configuration lives in config files, the web interface and run-time code
- Cluster types: single node, standalone, managed (YARN / Mesos / Kubernetes)
- **Driver**: one per cluster; assigns tasks, monitors and retries, consolidates results, manages shared data access
    - Size it with double the memory of other nodes and fast local storage
- **Worker**: runs assigned tasks and reports back; needs code, data and resources to hand or it pauses
    - More, smaller workers often beat fewer, larger ones (especially for import / export)
    - Fast local storage (SSD / NVMe) helps with caching and intermediate files
- Always test configurations: the fastest setup is not necessarily the best value

| Task | Syntax |
|---|---|
| Read a setting | `spark.conf.get("spark.app.name")` |
| Read with a default | `spark.conf.get("spark.driver.memory", "1g")` |
| Write a setting | `spark.conf.set("spark.sql.shuffle.partitions", 500)` |
| Check cluster type | `spark.conf.get("spark.master")` |
| Check partition count | `df.rdd.getNumPartitions()` |

In [28]:
# Name of the Spark application instance
app_name = spark.conf.get("spark.app.name")

# Driver TCP port
driver_tcp_port = spark.conf.get("spark.driver.port")

# Number of join partitions
num_partitions = spark.conf.get('spark.sql.shuffle.partitions')

# Show the results
print("Name: %s" % app_name)
print("Driver TCP port: %s" % driver_tcp_port)
print("Number of partitions: %s" % num_partitions)

Name: clean-data-w-spark
Driver TCP port: 45419
Number of partitions: 200


In [29]:
departures_df = spark.read.format("csv").options(header = True).load("AA_DFW_2017_Departures_Short.csv.gz").distinct()

In [30]:
# Store the number of partitions in departures_df in the variable before.
before = departures_df.rdd.getNumPartitions()

# Change the spark.sql.shuffle.partitions configuration to 500 partitions.
spark.conf.set("spark.sql.shuffle.partitions", 500)

# Recreate the departures_df DataFrame reading the distinct rows from the departures file.
departures_df = spark.read.csv("departures.txt.gz").distinct()

# Print the number of partitions from before and after the configuration change
print("Partition count before change: %d" % before)
print("Partition count after change: %d" % departures_df.rdd.getNumPartitions())

Partition count before change: 2
Partition count after change: 2


## Performance improvements


- Viewing the Spark execution plan with `.explain()`
- What shuffling is and why it slows things down
- Ways to limit shuffling (`.repartition()`, `.coalesce()`, careful joins)
- Broadcasting to speed up joins

### Explaining the Spark execution plan

- To understand performance implications, need to see what Spark is doing under the hood
- Easiest way: call `.explain()` on a DataFrame
- Returns the **estimated plan** that will be run to generate results from the DataFrame
- No need to understand the plan's specifics yet — just remember how to view it when needed

```python
# request a single column and run distinct against it
voter_df.select('name').distinct().explain()
```

### What is shuffling?

- Spark distributes data amongst the various nodes in the cluster
- **Shuffling** = the moving of data fragments to various workers as required to complete certain tasks
- Useful side: hides complexity from the user (no need to know which nodes hold what data)
- Downsides:
  - can be **slow**, especially if a few nodes need all the data
  - lowers overall **throughput** — workers sit waiting for data to transfer
  - ties up workers, limiting how many are free for remaining tasks
- Often necessary, but worth minimising as much as possible

### How to limit shuffling?

- Hard to remove shuffling entirely, but a few things help
- **`.repartition()`** — takes one argument, the number of partitions requested
  - requires a **full shuffle** of data between nodes & processes → quite costly

```python
departures_df = departures_df.repartition(8)
```

- **`.coalesce()`** — use this to **reduce** the number of partitions
  - takes a number **smaller** than the current count and consolidates data **without a full shuffle**
  - Note: calling `.coalesce()` with a **larger** number than current does nothing

```python
departures_df = departures_df.coalesce(2)
```

- **`.join()`** — powerful, but calling it indiscriminately often triggers shuffle operations → increased cluster load & slower processing
  - to avoid some shuffles when joining, use **`.broadcast()`** (more below)
- Key point: **optimise for what matters** — if the initial code's speed is already acceptable, time may be better spent elsewhere

### Broadcasting

- A method to provide a **copy of an object to each worker**
- With its own copy, each worker needs less communication between nodes → fewer data shuffles, more likely a node completes tasks independently
- Can drastically speed up `.join()` operations — especially when one DataFrame is **much smaller** than the other
- To use it:
  - import `broadcast` from `pyspark.sql.functions`
  - call `broadcast()` on the DataFrame to broadcast

```python
from pyspark.sql.functions import broadcast

combined_df = large_df.join(broadcast(small_df), on='id')
```

- Watch out:
  - broadcasting can **slow** operations with very small DataFrames
  - or if the **larger** DataFrame in a join is broadcast
  - Spark often optimises this automatically — but always test in the environment for best performance

### Key takeaways

- **See the plan:** `df.explain()` shows the estimated execution plan
- **Shuffling:** moving data between workers; necessary but slow — minimise it
- **Reduce partitions cheaply:** `.coalesce(n)` (smaller only, no full shuffle)
- **Change partitions fully:** `.repartition(n)` (full shuffle, costly)
- **Joins:** can cause shuffles; broadcast the **smaller** DataFrame to cut them
- **Broadcast cheat sheet:**

```python
from pyspark.sql.functions import broadcast
big_df.join(broadcast(small_df), on='key')
```

- **Golden rule:** optimise for what matters — don't tune code that's already fast enough; benchmark changes in your own environment

## Performance improvements

- Builds on earlier material around Spark clusters and improving import performance
- Focus here: general techniques to improve the performance of Spark tasks

### Explaining the Spark execution plan

- To understand the performance implications of any operation, it helps to see what Spark is doing under the hood
- Easiest way to inspect this: call `.explain()` on a DataFrame
- Returns the estimated plan Spark will run to generate the results
- No need to decode the specifics of the plan yet — the key thing is knowing how to view it when required

```python
# Request a single column and run distinct against it
distinct_df = df.select("column_name").distinct()

# View the estimated execution plan
distinct_df.explain()
```

### What is shuffling?

- Spark distributes data amongst the various nodes in the cluster
- **Shuffling** = the moving of data fragments between workers as required to complete certain tasks
- Upside: shuffling hides overall complexity — no need to know which nodes hold which data
- Downside: transfers can be slow, especially if a few nodes need all the data
- Shuffling lowers the overall throughput of the cluster — workers sit waiting for data to transfer
- This limits the number of workers available for the remaining tasks in the system
- Often necessary, but worth minimising as much as possible

### How to limit shuffling?

Fully removing shuffle operations is tricky, but a few things help limit it.

#### Repartition vs coalesce

- `.repartition(n)` takes a single argument — the number of partitions requested
- Previously used to illustrate the effect of partitions alongside `monotonically_increasing_id()`
- Requires a **full shuffle** of data between nodes and processes → quite costly
- To *reduce* partitions, prefer `.coalesce()` instead — consolidates data **without** a full shuffle
- Important: calling `.coalesce()` with a number **larger** than the current partition count does nothing

```python
# Costly — triggers a full shuffle
df_repart = df.repartition(10)

# Cheaper way to reduce partitions — no full shuffle
df_coalesced = df.coalesce(2)
```

#### Joins

- `.join()` is powerful, but calling it indiscriminately often causes shuffle operations → increased cluster load and slower processing
- To avoid some of the shuffle during joins, use `.broadcast()` (covered next)

#### Optimise for what matters

- Key point: remember that the speed of the initial code may already be perfectly acceptable
- Time may be better spent elsewhere rather than over-optimising data-cleaning operations

### Broadcasting

- **Broadcasting** = providing a copy of an object to each worker
- With its own copy of the data, each worker needs less communication with other nodes
- This limits data shuffles — a node is more likely to fulfil tasks independently
- Can drastically speed up `.join()` operations, especially when one DataFrame is **much smaller** than the other
- Implementation: import `broadcast` from `pyspark.sql.functions`, then call it with the DataFrame to broadcast

```python
from pyspark.sql.functions import broadcast

# Broadcast the smaller DataFrame in the join
result = large_df.join(broadcast(small_df), on="id", how="inner")
```

- Caution: broadcasting can **slow** operations with very small DataFrames, or if the **larger** DataFrame is broadcast
- Spark often optimises this automatically — but always run tests in the environment for best performance

### Key takeaways / cheat sheet

- **See the plan:** `df.explain()` shows the estimated execution plan
- **Shuffling** = data moved between workers; necessary but slow, so minimise it
- **Reduce partitions cheaply** with `.coalesce(n)`; `.repartition(n)` forces a full, costly shuffle
- Joins can trigger shuffles — mitigate with broadcasting
- **Broadcast** the *smaller* DataFrame only; never the larger one
- Optimise only where it matters — acceptable code may not be worth tuning

| Operation | Shuffle? | Use when |
|-----------|----------|----------|
| `.repartition(n)` | Full shuffle | Increasing **or** rebalancing partitions |
| `.coalesce(n)` | No full shuffle | **Reducing** partitions only |
| `broadcast(df)` | Avoids join shuffle | One DataFrame is much smaller |

In [31]:
flights_df = spark.read.format("csv").options(header=True).load("AA_DFW_2014_Departures_Short.csv.gz")
airports_df = spark.read.format("csv").options(header=True).load("airports_df.csv")

In [32]:
flights_df.printSchema()
airports_df.printSchema()

root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): string (nullable = true)

root
 |-- AIRPORTNAME: string (nullable = true)
 |-- IATA: string (nullable = true)



In [33]:
# Create a new DataFrame normal_df by joining flights_df with airports_df
normal_df = flights_df.join(airports_df, on = flights_df["Destination Airport"] == airports_df["IATA"])

# Determine which type of join is used in the query plan
normal_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [Destination Airport#972], [IATA#996], Inner, BuildRight, false
   :- Filter isnotnull(Destination Airport#972)
   :  +- FileScan csv [Date (MM/DD/YYYY)#970,Flight Number#971,Destination Airport#972,Actual elapsed time (Minutes)#973] Batched: false, DataFilters: [isnotnull(Destination Airport#972)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/mothma/dc-pyspark/clean-data-w-pyspark/AA_DFW_2014_Departur..., PartitionFilters: [], PushedFilters: [IsNotNull(Destination Airport)], ReadSchema: struct<Date (MM/DD/YYYY):string,Flight Number:string,Destination Airport:string,Actual elapsed ti...
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[1, string, false]),false), [plan_id=1455]
      +- Filter isnotnull(IATA#996)
         +- FileScan csv [AIRPORTNAME#995,IATA#996] Batched: false, DataFilters: [isnotnull(IATA#996)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/mothma/

In [34]:
# Import the broadcast() method from pyspark.sql.functions
from pyspark.sql.functions import broadcast

# Create a new DataFrame broadcast_df by joining flights_df with airports_df, using the broadcasting
broadcast_df = flights_df.join(broadcast(airports_df), \
    flights_df["Destination Airport"] == airports_df["IATA"] )

# Show the query plan and consider differences from the original
broadcast_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [Destination Airport#972], [IATA#996], Inner, BuildRight, false
   :- Filter isnotnull(Destination Airport#972)
   :  +- FileScan csv [Date (MM/DD/YYYY)#970,Flight Number#971,Destination Airport#972,Actual elapsed time (Minutes)#973] Batched: false, DataFilters: [isnotnull(Destination Airport#972)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/mothma/dc-pyspark/clean-data-w-pyspark/AA_DFW_2014_Departur..., PartitionFilters: [], PushedFilters: [IsNotNull(Destination Airport)], ReadSchema: struct<Date (MM/DD/YYYY):string,Flight Number:string,Destination Airport:string,Actual elapsed ti...
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[1, string, false]),false), [plan_id=1478]
      +- Filter isnotnull(IATA#996)
         +- FileScan csv [AIRPORTNAME#995,IATA#996] Batched: false, DataFilters: [isnotnull(IATA#996)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/mothma/

In [35]:
start_time = time.time()
# Execute .count() on the normal DataFrame
normal_count = normal_df.count()
normal_duration = time.time() - start_time

start_time = time.time()
# Execute .count() on the broadcasted DataFrame
broadcast_count = broadcast_df.count()
broadcast_duration = time.time() - start_time

# Print the count and duration of the DataFrames noting and differences
print("Normal count:\t\t%d\tduration: %f" % (normal_count, normal_duration))
print("Broadcast count:\t%d\tduration: %f" % (broadcast_count, broadcast_duration))

Normal count:		156308	duration: 1.090770
Broadcast count:	156308	duration: 0.716299


#IV

## Introduction to Data Pipelines


- Most of the course so far: individual transformations and actions to clean data in Spark
- Reality check — data is rarely simple enough that a couple of transformations or actions can prepare it for real analysis
- Moving on to **data pipelines**

### What is a data pipeline?

- Simply the set of steps needed to move from an input data source (or sources) and convert it to the desired output
- Can consist of any number of steps or components, and can span many systems
- Here: setting up a pipeline within Spark — but a full production pipeline will likely communicate with many systems

### What does a data pipeline look like?

Much like Spark in general, a pipeline typically consists of **inputs → transformations → outputs**, often with **validation** and **analysis** steps before delivery.

#### Inputs

- Any data type covered so far — CSV, JSON, text, etc.
- Source could be the local file system, web services, APIs, databases, and so on
- Basic idea: read the data into a DataFrame

```python
schema = StructType([
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), False)
])

df = spark.read.format("csv").schema(schema).load("input_data.csv")
```

#### Transformations

- The individual steps done throughout the course — adding columns, filtering rows, performing calculations as needed
- Previous examples used only one or two steps at a time; a pipeline can chain as many as needed to format the data into the desired output

```python
from pyspark.sql.functions import col

transformed_df = (df
    .filter(col("age") > 18)
    .withColumn("age_group", col("age") / 10)
)
```

#### Outputs

- Output the data into a usable form
- Already written files out to CSV or Parquet
- Could also mean multiple copies in various formats, or writing to a database, a web service, etc.

```python
transformed_df.write.format("parquet").save("output_data.parquet")
```

#### Validation

- Covered in a later lesson
- Idea: run some form of testing on the data to verify it is as expected

#### Analysis

- Often the final step before handing the data off to the next user
- Can include row counts, specific calculations, or anything that makes the dataset easier to consume

```python
print("Row count:", transformed_df.count())
```

- Important to remember: the last two steps (validation and analysis) vary greatly depending on needs

### Pipeline details

- Key point: in Spark, a data pipeline is **not a formally defined object** — it is a *concept*
- Different from the `Pipeline` object in Spark.ML (not needed for this course)
- For these purposes, a Spark pipeline is all the normal code required to complete a task
- Example flow: define a schema → read a datafile → add an ID → write out two separate data types
- The task could be much more complex, but the concept is usually the same

```python
from pyspark.sql.functions import monotonically_increasing_id

# Define schema, read file, add an ID, then write out two formats
schema = StructType([
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), False)
])

df = spark.read.format("csv").schema(schema).load("datafile.csv")
df = df.withColumn("id", monotonically_increasing_id())

df.write.format("csv").save("output.csv")
df.write.format("parquet").save("output.parquet")
```

### Key takeaways / cheat sheet

- A **data pipeline** = the steps that take input data and convert it into the desired output
- Standard shape: **input → transform → output**, plus optional **validation** and **analysis**
- In Spark, a pipeline is a *concept*, not a formal object (unlike Spark.ML's `Pipeline`)
- Inputs: `spark.read.format(...).schema(...).load(...)`
- Transformations chain freely — `.filter()`, `.withColumn()`, calculations, etc.
- Outputs: `df.write.format("csv" | "parquet").save(...)`, and can target databases or services too
- Validation = test the data is as expected; analysis = counts/calculations to help the next user

In [43]:
df18 = spark.read.format("csv").options(header=True).load("AA_DFW_2015_Departures_Short.csv.gz")

df18.write.format("csv").options(header=True).mode("overwrite").save("2015-departures.csv.gz")

In [44]:
df18.printSchema()

root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): string (nullable = true)



In [46]:
import pyspark.sql.functions as F

# Import the file 2015-departures.csv.gz to a DataFrame. Note the header is already defined
departures_df = spark.read.csv("2015-departures.csv.gz", header=True)

# Filter the DataFrame to contain only flights with a duration over 0 minutes. Use the index of the column, not the column name (remember to use .printSchema() to see the column names / order)
departures_df = departures_df.filter(departures_df["Actual elapsed time (Minutes)"] > 0)

# Add an ID column
departures_df = departures_df.withColumn('id', F.monotonically_increasing_id())

# Write the file out as a JSON document named output.json
departures_df.write.json("output.json", mode="overwrite")

## Data handling techniques


- Builds on earlier data-cleaning operations in Spark
- Focus here: using the methods already learned to parse **unconventional** data

### What are we trying to parse?

- When reading data into Spark, a fully uniform file is rare — there's often content that needs removing or reformatting
- Common issues:
  - **Incorrect data** — empty rows, commented lines, headers, or rows that don't match the intended schema
  - **Nested structures** — columns using *different* delimiters (e.g. primary columns split by a comma, but some components split by a semi-colon)
  - **Non-tabular data** — sometimes a differing number of columns per row
- Many ways to parse these situations; the right one depends on specific needs
- Course focuses on CSV, but the general scenarios apply to other formats too

### Stanford ImageNet annotations

- Dataset for this chapter: Stanford ImageNet annotations — finding and identifying **dogs** in ImageNet images
- Provides a list of all identified dogs in an image, including when multiple dogs appear in the same image
- Metadata included: the folder within the ImageNet dataset, the image dimensions, and the bounding box(es) of the dog(s)
- Example row layout: folder name, ImageNet image reference, width, height — then image data per dog
- Each breed "column" = the breed name **plus** the bounding box in the image
- Example quirk: row 1 has one Newfoundland; row 2 has **two** Bull Mastiffs and therefore an **additional** "column"
- Key point: a varying number of columns per row is exactly the kind of non-tabular data to handle

### Removing blank lines, headers, and comments

Spark's CSV parser handles many common issues via optional parameters.

- **Blank lines** — automatically removed when using the CSV parser (unless specifically instructed otherwise)
- **Comments** — removed with the optional `comment` argument, specifying the character a comment line starts with
  - Note: handles lines that *begin* with the comment character; more complex comment usage needs more involved procedures
- **Header rows** — parsed via the optional `header` parameter, set to `True` or `False`
  - No schema defined → column names initially set from the header
  - Schema defined → the row isn't used as data, and the header names are otherwise ignored

```python
# Skip comment lines starting with '#' and treat first row as a header
df = spark.read.format("csv") \
    .option("comment", "#") \
    .option("header", "True") \
    .load("annotations.csv")
```

### Automatic column creation

- On import, Spark automatically creates DataFrame columns where it can
- Splits each row of text on the separator argument `sep`
- If `sep` isn't defined → defaults to a **comma**
- The parser still succeeds even if the separator character isn't within the string — it stores the **entire row** in a single column named `_c0` by default
- Trick: forcing everything into `_c0` allows parsing of nested or complex data (more on this later)

```python
# Force the whole row into a single column _c0 by using an unlikely separator
df = spark.read.format("csv") \
    .option("sep", "*") \
    .load("complex_data.csv")

# _c0 now holds the full raw row for custom parsing
df.select("_c0").show(truncate=False)
```

### Key takeaways / cheat sheet

- Real-world files are rarely uniform — expect blank rows, comments, headers, mixed delimiters and non-tabular shapes
- `.option("comment", "#")` → drop lines starting with that character
- `.option("header", "True"/"False")` → use (or ignore) the first row as column names; a defined schema overrides header names
- Blank lines are removed automatically by the CSV parser
- `.option("sep", ...)` → set the column separator (defaults to `,`)
- Pick a separator *not* present in the data to dump the whole row into `_c0` → handy for nested/complex parsing

| Parameter | Purpose |
|-----------|---------|
| `header` | Treat first row as column names (`True`/`False`) |
| `comment` | Character marking comment lines to remove |
| `sep` | Column separator (default `,`) |
| `schema` | Enforce column names/types; ignores header names |

In [47]:
from pyspark.sql.functions import col

# Import the annotations.csv.gz file to a DataFrame and perform a row count. Specify a separator character of |.
annotations_df = spark.read.csv('annotations.csv.gz', sep="|")
full_count = annotations_df.count()

# Query the data for the number of rows beginning with #
comment_count = annotations_df.filter(col('_c0').startswith('#')).count()

# Import the file again to a new DataFrame, but specify the comment character in the options to remove any commented rows
no_comments_df = spark.read.csv("annotations.csv.gz", sep = "|", comment = "#")

# # Count the new DataFrame and verify the difference is as expected
no_comments_count = no_comments_df.count()
print("Full count: %d\nComment count: %d\nRemaining count: %d" % (full_count, comment_count,no_comments_count))

Full count: 20650
Comment count: 40
Remaining count: 20610


In [48]:
initial_count = no_comments_count
annotations_df = no_comments_df.select("*")


In [49]:
# Create a new variable tmp_fields using the annotations_df DataFrame column '_c0' splitting it on the tab character
tmp_fields = F.split(annotations_df['_c0'], "\t")

# # Create a new column in annotations_df named 'colcount' representing the number of fields defined in the previous step
annotations_df = annotations_df.withColumn('colcount', F.size(tmp_fields))

# # Filter out any rows from annotations_df containing fewer than 5 fields
annotations_df_filtered = annotations_df.filter(~ (annotations_df['colcount'] < 5))

# # Count the number of rows in the DataFrame and compare to the initial_count
final_count = annotations_df_filtered.count()
print("Initial count: %d\nFinal count: %d" % (initial_count, final_count))

Initial count: 20610
Final count: 20585


In [50]:
annotations_df = annotations_df_filtered.select("*")

In [51]:
# Split the content of _c0 on the tab character (aka, '\t')
split_cols = F.split(annotations_df['_c0'], '\t')

# Add the columns folder, filename, width, and height
split_df = annotations_df.withColumn('folder', split_cols.getItem(0))
split_df = split_df.withColumn('filename', split_cols.getItem(1))
split_df = split_df.withColumn('width', split_cols.getItem(2))
split_df = split_df.withColumn('height', split_cols.getItem(3))

# Add split_cols as a column
split_df = split_df.withColumn('split_cols', split_cols)

In [52]:
def retriever(cols, colcount):
  # Return a list of dog data
  return cols[4:colcount]

# Define the method as a UDF
udfRetriever = F.udf(retriever, ArrayType(StringType()))

# Create a new column using your UDF
split_df = split_df.withColumn('dog_list', udfRetriever(split_df.split_cols, split_df.colcount))

# Remove the original column, split_cols, and the colcount
split_df = split_df.drop('_c0').drop('split_cols').drop('colcount')

## Data validation


- Validation is a data pipeline step that verifies the quality of data before it is delivered
- Focus: implementing validation steps within a data cleaning pipeline

### Definition

- Validation = verifying that a dataset complies with an expected format
- Common checks:
  - **Row/column counts as expected** — e.g. is the row count within 2% of the previous month's row count?
  - **Data types match** the expectation
  - **Content requirements** where no schema is specifically enforced — e.g. a field must be 9 characters or fewer
  - **Complex rules** — e.g. a set of sensor readings must fall within physically possible quantities

### Validating via joins

- A join-based technique compares the content of a DataFrame against a known set of values (known ids, companies, addresses, etc.)
- Joins make it easy to determine whether data is present in a set:
  - rows present in only one DataFrame
  - rows present in both
  - rows present in neither
- Comparatively fast — much quicker than validating individual rows against a long list of entries

**Simplest example — inner join:**

- `parsed_df` is loaded from a parquet file
- A second DataFrame holds a list of known company names
- A new DataFrame is created by joining the two on the company name
- Because it is an *inner* join, only rows from `parsed_df` whose company names also appear in `company_df` survive into the result (`verified_df`)
- Effect: rows that don't meet the criteria are filtered out automatically — no explicit Spark filter or comparison code required

```python
# parsed data loaded from parquet
parsed_df = spark.read.parquet("/data/parsed_data.parquet")

# known-good reference set of company names
company_df = spark.read.parquet("/data/known_companies.parquet")

# inner join keeps only rows whose company exists in the reference set
verified_df = parsed_df.join(company_df, on="company", how="inner")
```

### Complex rule validation

- Using Spark components to validate logic
- Can be as simple as using Spark calculations to verify the number of columns in an irregular dataset
- Validation can also be run against an **external source** — a web service, local files, or API calls
- Such rules are often implemented as a **UDF**, encapsulating the logic in one place so it can be run easily against the content of a DataFrame

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

# rule: a sensor reading must sit within a physically possible range
def within_range(reading):
    return reading is not None and 0 <= reading <= 100

within_range_udf = udf(within_range, BooleanType())

validated_df = (
    sensor_df
    .withColumn("is_valid", within_range_udf("reading"))
    .filter("is_valid")
)
```

### Key takeaways / cheat sheet

- **Validation** = confirming a dataset matches an expected format (counts, types, content rules, complex logic)
- **Validate via joins** when checking content against a known reference set — fast and filter-free:

| Goal | Join type |
|---|---|
| Keep only rows present in the reference set | `how="inner"` |
| Find rows **not** in the reference set | `how="left_anti"` |
| Keep all rows, flag matches | `how="left"` |

```python
# keep matches
df.join(ref_df, on="key", how="inner")

# find rows missing from the reference set
df.join(ref_df, on="key", how="left_anti")
```

- **Complex rules** → wrap the logic in a UDF and apply it across the DataFrame; can reach out to external sources (web service, files, API)
- Preferred because the logic lives in **one place** and runs consistently against every row

In [54]:
from pyspark.sql import functions as F

valid_folders_df = spark.read.csv("valid_folders.txt")

In [ ]:

# # Rename the _c0 column to folder on valid_folders_df
# valid_folders_df = valid_folders_df.withColumnRenamed('c0', 'folder')

# # Count the number of rows in split_df
# split_count = 

# # Join split_df with the broadcast of valid_folders_df on folder
# joined_df = split_df.join(F.broadcast(valid_folders_df), "folder")

# # Count the number of rows remaining in joined_df and compare
# joined_count = joined_df.count()
# print("Before: %d\nAfter: %d" % (split_count, joined_count))

In [57]:
# # Determine the row counts for each DataFrame
# split_count = split_df.count()
# joined_count = joined_df.count()

# # Create a DataFrame containing the invalid rows
# invalid_df = split_df.join(F.broadcast(joined_df), 'folder', 'anti')

# # Validate the count of the new DataFrame is as expected
# invalid_count = invalid_df.count()
# print(" split_df:\t%d\n joined_df:\t%d\n invalid_df: \t%d" % (split_count, joined_count, invalid_count))

# # Determine the number of distinct folder rows removed
# invalid_folder_count = invalid_df.select('folder').distinct().count()
# print("%d distinct invalid folders found" % invalid_folder_count)

## Final analysis and delivery


- Finishing the data cleaning pipeline with final analysis calculations
- Builds on the Spark components already used throughout data cleaning

### Analysis calculations (UDF)

- Analysis calculations = using the columns of a DataFrame to compute some useful value via Spark's functionality
- Example here: calculating an average sale price from a given list of sales
- A Python function takes a `saleslist` argument:
  - for every sale in the list, it adds the sale entry (drawn from values 2 and 3 in the sale tuple)
  - once complete, it works out the average per row and returns it
- The remainder is the usual pattern for defining a UDF and applying it within a DataFrame

```python
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

# compute average sale price from a list of sale tuples
def getAvgSale(saleslist):
    total = 0
    count = len(saleslist)
    for sale in saleslist:
        total += sale[2] + sale[3]
    return total / count

udfGetAvgSale = udf(getAvgSale, FloatType())

df = df.withColumn("avg_sale", udfGetAvgSale(df.sales_list))
```

### Analysis calculations (inline)

- UDFs are powerful and flexible, and are sometimes the only way to handle certain data
- **Drawback:** UDFs carry a performance penalty compared with built-in Spark functions, especially for some operations
- **Solution:** perform calculations inline wherever possible
- Spark columns can be defined using inline math operations, which Spark can then optimise for best performance
- Worked examples (after reading in a datafile):
  - a simple average computed from two existing columns
  - a square footage computed by multiplying two columns together to form a third
  - a mix of a UDF with an inline calculation — often a better approach exists, but it shows Spark doesn't care about the source of the information so long as it conforms to the expected input format

```python
# inline average of two columns
df = df.withColumn("avg", (df.total_sales / df.sale_count))

# inline square footage by multiplying two columns
df = df.withColumn("sq_ft", df.width * df.length)

# mixing a UDF with an inline calculation
df = df.withColumn("price_sqft", udfGetAvgSale(df.sales_list) / df.sq_ft)
```

### Key takeaways / cheat sheet

- **Analysis calculations** turn existing columns into useful derived values
- **UDF route** — flexible, sometimes unavoidable, but slower than built-ins:

```python
my_udf = udf(python_func, ReturnType())
df = df.withColumn("new_col", my_udf(df.some_col))
```

- **Inline route** — prefer this when possible; built-in column maths is optimised by Spark:

```python
df = df.withColumn("new_col", df.col_a / df.col_b)   # division
df = df.withColumn("area", df.col_a * df.col_b)       # multiplication
```

| Approach | Speed | When to use |
|---|---|---|
| Inline column maths | Fast (optimised) | Default — arithmetic on columns |
| UDF | Slower | Custom logic built-ins can't express |

- Spark accepts input from any source — UDF output, inline result, or a mix — as long as it matches the expected input format

In [90]:
joined_df = (
    spark.read.csv("joined_df.csv", header=True, escape='"')
    .withColumn("dog_list", F.from_json("dog_list", "array<string>"))
)

joined_df.printSchema()

root
 |-- folder: string (nullable = true)
 |-- filename: string (nullable = true)
 |-- width: string (nullable = true)
 |-- height: string (nullable = true)
 |-- dog_list: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [91]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Select the dog details column and show the first 10 un-truncated rows
print(joined_df.select("dog_list").show(n=10, truncate=False))

# # Create a schema with bred, start_x, start_y, end_x, end_y (numbers are integers)
DogType = StructType([
	StructField("breed", IntegerType(), False),
    StructField("start_x", IntegerType(), False),
    StructField("start_y", IntegerType(), False),
    StructField("end_x", IntegerType(), False),
    StructField("end_y", IntegerType(), False),
])

+----------------------------------+
|dog_list                          |
+----------------------------------+
|[affenpinscher,0,9,173,298]       |
|[Border_terrier,73,127,341,335]   |
|[kuvasz,0,0,499,327]              |
|[Great_Pyrenees,124,225,403,374]  |
|[schipperke,146,29,416,309]       |
|[groenendael,168,0,469,374]       |
|[Bedlington_terrier,10,12,462,332]|
|[Lhasa,39,1,499,373]              |
|[Kerry_blue_terrier,17,16,300,482]|
|[vizsla,112,93,276,236]           |
+----------------------------------+
only showing top 10 rows

None


In [92]:
from pyspark.sql.functions import udf

# Split each entry in dog_list into its parts, converting strings to the correct types
def dogParse(doglist):
  dogs = []
  for dog in doglist:
    (breed, start_x, start_y, end_x, end_y) = dog.split(',')
    dogs.append((breed, int(start_x), int(start_y), int(end_x), int(end_y)))
  return dogs

# Create a UDF from the above function
udfDogParse = udf(dogParse, ArrayType(DogType))

# Use the UDF to create a new column called dogs
joined_df = joined_df.withColumn('dogs', udfDogParse('dog_list'))

# # Show the number of dogs in the new column for the first 10 rows
joined_df.select(F.size('dogs')).show(10)

+----------+
|size(dogs)|
+----------+
|         1|
|         1|
|         1|
|         1|
|         1|
|         1|
|         1|
|         1|
|         1|
|         1|
+----------+
only showing top 10 rows



In [94]:
# Define a Python function to take a list of tuples (the dog objects) and calculate the total number of "dog" pixels per image.
def dogPixelCount(doglist):
  totalpixels = 0
  for dog in doglist:
    totalpixels += (dog[3] - dog[1]) * (dog[4] - dog[2])
  return totalpixels

# Create a UDF of the function
udfDogPixelCount = udf(dogPixelCount, IntegerType())

# use it to create a new column called 'dog_pixels' on the DataFrame.
joined_df = joined_df.withColumn('dog_pixels', udfDogPixelCount('dogs'))

# Create another column, 'dog_percent', representing the percentage of 'dog_pixels' in the image. Make sure this is between 0-100%. Use the string name of the column alone (ie, "columnname" rather than df.columnname).
joined_df = joined_df.withColumn('dog_percent', (F.col('dog_pixels') / (F.col('width') * F.col('height'))) * 100)

# Show the first 10 rows with more than 60% 'dog_pixels' in the image. Use a SQL style string for this (ie, 'columnname > ____').
joined_df.where("dog_percent > 60").show(10)

+--------+---------------+-----+------+--------------------+--------------------+----------+-----------------+
|  folder|       filename|width|height|            dog_list|                dogs|dog_pixels|      dog_percent|
+--------+---------------+-----+------+--------------------+--------------------+----------+-----------------+
|02110627|n02110627_12938|  200|   300|[affenpinscher,0,...|[{0, 0, 9, 173, 2...|     49997|83.32833333333333|
|02104029|   n02104029_63|  500|   375|[kuvasz,0,0,499,327]|[{0, 0, 0, 499, 3...|    163173|          87.0256|
|02105056| n02105056_2834|  500|   375|[groenendael,168,...|[{0, 168, 0, 469,...|    112574|60.03946666666666|
|02093647|  n02093647_541|  500|   333|[Bedlington_terri...|[{0, 10, 12, 462,...|    144640|86.87087087087087|
|02098413| n02098413_1355|  500|   375|[Lhasa,39,1,499,373]|[{0, 39, 1, 499, ...|    171120|           91.264|
|02093859| n02093859_2309|  330|   500|[Kerry_blue_terri...|[{0, 17, 16, 300,...|    131878|79.92606060606062|
|

In [82]:
joined_df.show()

26/11/20 20:40:03 ERROR Executor: Exception in task 0.0 in stage 148.0 (TID 715)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/tmp/ipykernel_2099/4145101151.py", line 5, in dogPixelCount
TypeError: unsupported operand type(s) for -: 'str' and 'str'

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:94)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:37)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:491)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/tmp/ipykernel_2099/4145101151.py", line 5, in dogPixelCount
TypeError: unsupported operand type(s) for -: 'str' and 'str'
